# SARIMA

## 1. Configuration & Setup

In [1]:
# Check dependencies
import sys

required_packages = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'xarray': 'xarray',
    'netCDF4': 'netCDF4',
    'matplotlib': 'matplotlib',
    'torch': 'torch',
    'statsmodels': 'statsmodels',
    'tqdm': 'tqdm'
}

missing_packages = []
for package_name, import_name in required_packages.items():
    try:
        __import__(import_name)
        print(f"✓ {package_name} is installed")
    except ImportError as e:
        print(f"✗ {package_name} is NOT installed - {str(e)}")
        missing_packages.append(package_name)

if missing_packages:
    print(f"\n⚠️  Missing packages: {', '.join(missing_packages)}")
else:
    print("\n✓ All required packages are installed!")

✓ numpy is installed
✓ pandas is installed
✓ xarray is installed
✗ netCDF4 is NOT installed - No module named 'netCDF4'
✓ matplotlib is installed
✓ torch is installed
✓ statsmodels is installed
✓ tqdm is installed

⚠️  Missing packages: netCDF4


In [2]:
# !pip install netCDF4

In [3]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
#%%
# ============== META VARIABLES ==============
import os
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from math import sqrt
import warnings
warnings.filterwarnings('ignore')

import statsmodels.api as sm
import statsmodels.formula.api as smf

# Set random seeds for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Data paths
DATA_DIR = "drive/MyDrive/Colab Notebooks/MLPS_proj/data"
RESULTS_DIR = "drive/MyDrive/Colab Notebooks/MLPS_proj/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

TRAIN_PATH = os.path.join(DATA_DIR, "ds_train_clean.nc")

# The test_24h_demo.nc and test_48h_demo.nc are just demo files, so the outages in those files are just noise
# TEST_24H_PATH = os.path.join(DATA_DIR, "test_24h.nc")
# TEST_48H_PATH = os.path.join(DATA_DIR, "test_48h.nc")
TEST_24H_PATH = os.path.join(DATA_DIR, "test_24h_demo.nc")
TEST_48H_PATH = os.path.join(DATA_DIR, "test_48h_demo.nc")

# Model parameters
VALIDATION_SPLIT = 0.2  # Use last 20% of training data for validation

# SARIMA parameters
SARIMA_ORDER = (1, 0, 1)  # (p, d, q)

# Seq2Seq parameters
SEQ_LEN = 24       # Lookback window (hours) for the seq2seq model
BATCH_SIZE = 64
EPOCHS = 5
LEARNING_RATE = 1e-3
HIDDEN_DIM = 64
NUM_LAYERS = 1

# Set device for PyTorch
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Configuration loaded successfully!")
print(f"Random Seed: {RANDOM_SEED}")
print(f"Device: {DEVICE}")
print(f"Data Directory: {DATA_DIR}")
print(f"Results Directory: {RESULTS_DIR}")

Configuration loaded successfully!
Random Seed: 42
Device: cpu
Data Directory: drive/MyDrive/Colab Notebooks/MLPS_proj/data
Results Directory: drive/MyDrive/Colab Notebooks/MLPS_proj/results


## 2. Data Loading

In [5]:
# Load datasets
ds_train = xr.open_dataset(TRAIN_PATH)

print(ds_train)

<xarray.Dataset> Size: 159MB
Dimensions:    (location: 83, timestamp: 2161, feature: 109)
Coordinates:
  * location   (location) <U5 2kB '26001' '26003' '26005' ... '26163' '26165'
  * timestamp  (timestamp) datetime64[ns] 17kB 2023-04-01 ... 2023-06-30
  * feature    (feature) object 872B 'SBT113' 'SBT114' 'SBT123' ... 'wz' 'wz_1'
    state      (location) <U2 664B ...
Data variables:
    weather    (location, timestamp, feature) float64 156MB ...
    tracked    (location, timestamp) float64 1MB ...
    out        (location, timestamp) float64 1MB ...


## 3. Basic SARIMA From Demo

In [6]:
# Extract basic information
train_timestamps = pd.to_datetime(ds_train.timestamp.values)
locations = list(ds_train.location.values)
weather_features = list(ds_train.feature.values) if 'feature' in ds_train.dims else []

print(f"Training Period: {train_timestamps.min()} to {train_timestamps.max()}")
print(f"Number of Timestamps: {len(train_timestamps)}")
print(f"Number of Locations: {len(locations)}")
print(f"Locations: {locations}")
print(f"\nWeather Features ({len(weather_features)}): {weather_features}")

# Extract outage data
outage_data = ds_train.out.transpose("timestamp", "location").values.astype(float)
print(f"\nOutage Data Shape: {outage_data.shape} (timestamps x locations)")
print(f"Outage Statistics:")
print(f"  Mean: {np.nanmean(outage_data):.2f}")
print(f"  Std: {np.nanstd(outage_data):.2f}")
print(f"  Min: {np.nanmin(outage_data):.2f}")
print(f"  Max: {np.nanmax(outage_data):.2f}")

Training Period: 2023-04-01 00:00:00 to 2023-06-30 00:00:00
Number of Timestamps: 2161
Number of Locations: 83
Locations: [np.str_('26001'), np.str_('26003'), np.str_('26005'), np.str_('26007'), np.str_('26009'), np.str_('26011'), np.str_('26013'), np.str_('26015'), np.str_('26017'), np.str_('26019'), np.str_('26021'), np.str_('26023'), np.str_('26025'), np.str_('26027'), np.str_('26029'), np.str_('26031'), np.str_('26033'), np.str_('26035'), np.str_('26037'), np.str_('26039'), np.str_('26041'), np.str_('26043'), np.str_('26045'), np.str_('26047'), np.str_('26049'), np.str_('26051'), np.str_('26053'), np.str_('26055'), np.str_('26057'), np.str_('26059'), np.str_('26061'), np.str_('26063'), np.str_('26065'), np.str_('26067'), np.str_('26069'), np.str_('26071'), np.str_('26073'), np.str_('26075'), np.str_('26077'), np.str_('26079'), np.str_('26081'), np.str_('26083'), np.str_('26085'), np.str_('26087'), np.str_('26089'), np.str_('26091'), np.str_('26093'), np.str_('26095'), np.str_('2609

In [7]:
ds_test_24h = xr.open_dataset(TEST_24H_PATH)
ds_test_48h = xr.open_dataset(TEST_48H_PATH)

In [8]:
# Load test datasets
print("Loading test datasets...")

test_24h_timestamps = ds_test_24h.timestamp.values
test_48h_timestamps = ds_test_48h.timestamp.values

print(f"✓ Test 24h: {len(test_24h_timestamps)} timestamps")
print(f"✓ Test 48h: {len(test_48h_timestamps)} timestamps")

print(f"\nTesting Period (24h): {test_24h_timestamps.min()} to {test_24h_timestamps.max()}")
print(f"Testing Period (48h): {test_48h_timestamps.min()} to {test_48h_timestamps.max()}")

Loading test datasets...
✓ Test 24h: 24 timestamps
✓ Test 48h: 48 timestamps

Testing Period (24h): 2023-06-30T01:00:00.000000000 to 2023-07-01T00:00:00.000000000
Testing Period (48h): 2023-06-30T01:00:00.000000000 to 2023-07-02T00:00:00.000000000


### 3.1 Data Preparation

Split the training data into train and validation sets for model selection.

In [9]:
# Create validation split (temporal split)
n_timestamps = len(train_timestamps)
split_idx = int(n_timestamps * (1 - VALIDATION_SPLIT))

# Split the dataset
ds_train_sub = ds_train.isel(timestamp=slice(0, split_idx))
ds_val = ds_train.isel(timestamp=slice(split_idx, None))

train_sub_timestamps = pd.to_datetime(ds_train_sub.timestamp.values)
val_timestamps = pd.to_datetime(ds_val.timestamp.values)

print(f"Training Subset: {len(train_sub_timestamps)} timestamps")
print(f"  Period: {train_sub_timestamps.min()} to {train_sub_timestamps.max()}")
print(f"\nValidation Set: {len(val_timestamps)} timestamps")
print(f"  Period: {val_timestamps.min()} to {val_timestamps.max()}")

# Store validation ground truth for later evaluation
val_truth = ds_val.out.transpose("timestamp", "location").values.astype(float)

Training Subset: 1728 timestamps
  Period: 2023-04-01 00:00:00 to 2023-06-11 23:00:00

Validation Set: 433 timestamps
  Period: 2023-06-12 00:00:00 to 2023-06-30 00:00:00


In [10]:
val_timestamps

DatetimeIndex(['2023-06-12 00:00:00', '2023-06-12 01:00:00',
               '2023-06-12 02:00:00', '2023-06-12 03:00:00',
               '2023-06-12 04:00:00', '2023-06-12 05:00:00',
               '2023-06-12 06:00:00', '2023-06-12 07:00:00',
               '2023-06-12 08:00:00', '2023-06-12 09:00:00',
               ...
               '2023-06-29 15:00:00', '2023-06-29 16:00:00',
               '2023-06-29 17:00:00', '2023-06-29 18:00:00',
               '2023-06-29 19:00:00', '2023-06-29 20:00:00',
               '2023-06-29 21:00:00', '2023-06-29 22:00:00',
               '2023-06-29 23:00:00', '2023-06-30 00:00:00'],
              dtype='datetime64[ns]', length=433, freq=None)

In [11]:
# Prepare ground truth for last 24 and 48 hours to align with test set
val_truth_24h = ds_val.out.transpose("timestamp", "location").isel(timestamp=slice(0, 24)).values.astype(float)
val_truth_48h = ds_val.out.transpose("timestamp", "location").isel(timestamp=slice(0, 48)).values.astype(float)

print(f"\Validation set shapes:")
print(f"  24h: {val_truth_24h.shape}")
print(f"  48h: {val_truth_48h.shape}")

\Validation set shapes:
  24h: (24, 83)
  48h: (48, 83)


In [12]:
test_24h_truth = ds_test_24h.out.transpose("timestamp", "location").values.astype(float)
test_48h_truth = ds_test_48h.out.transpose("timestamp", "location").values.astype(float)


print(f"\nTest shapes:")
print(f"  24h: {test_24h_truth.shape}")
print(f"  48h: {test_48h_truth.shape}")


Test shapes:
  24h: (24, 83)
  48h: (48, 83)


### 3.2 SARIMA

In [13]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

def safe_fit_sarima(y, order=(1, 0, 1)):
    """Safely fit SARIMA model with error handling."""
    y = np.asarray(y, dtype=float).flatten()
    if len(y) < 8 or np.allclose(y, y[0]):
        return None
    try:
        model = SARIMAX(y, order=order, enforce_stationarity=False, enforce_invertibility=False)
        res = model.fit(disp=False)
        return res
    except Exception as e:
        print(f"  Warning: SARIMA fit failed - {str(e)[:50]}")
        return None

### 3.3 Prediction Functions

Define reusable prediction functions that will be used for both validation and test sets.

In [14]:
# Consolidated prediction functions for reuse

def generate_sarima_predictions(models_dict, locations, timestamps):
    """
    Generate SARIMA predictions for given timestamps.
    Returns: DataFrame in long format (timestamp, location, pred)
    """
    rows = []
    n_steps = len(timestamps)

    for loc in locations:
        loc_str = str(loc)
        if loc_str in models_dict and models_dict[loc_str] is not None:
            try:
                pred = np.asarray(models_dict[loc_str].forecast(steps=n_steps), dtype=float)
                pred = np.clip(pred, 0, None)  # Non-negative constraint
            except:
                pred = np.zeros(n_steps)
        else:
            pred = np.zeros(n_steps)

        rows.append(pd.DataFrame({
            "timestamp": timestamps,
            "location": loc_str,
            "pred": pred
        }))

    return pd.concat(rows, ignore_index=True)

### 3.6 Evaluation Metric Fucntions

In [15]:
# Define evaluation metrics
def rmse(y_true, y_pred):
    """Calculate Root Mean Squared Error."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

# def mae(y_true, y_pred):
#     """Calculate Mean Absolute Error."""
#     y_true = np.asarray(y_true, dtype=float)
#     y_pred = np.asarray(y_pred, dtype=float)
#     return float(np.mean(np.abs(y_true - y_pred)))

### 3.7 Validation Set (24h and 48h)

Evaluate models on 24h and 48h prediction horizons separately, matching the test set structure.

In [16]:
# Define horizons
horizon_24h_val = 24
horizon_48h_val = 48

# Extract validation timestamps for each horizon
val_timestamps_24h = val_timestamps[:24]
val_timestamps_48h = val_timestamps[:48]

val_timestamps_48h


DatetimeIndex(['2023-06-12 00:00:00', '2023-06-12 01:00:00',
               '2023-06-12 02:00:00', '2023-06-12 03:00:00',
               '2023-06-12 04:00:00', '2023-06-12 05:00:00',
               '2023-06-12 06:00:00', '2023-06-12 07:00:00',
               '2023-06-12 08:00:00', '2023-06-12 09:00:00',
               '2023-06-12 10:00:00', '2023-06-12 11:00:00',
               '2023-06-12 12:00:00', '2023-06-12 13:00:00',
               '2023-06-12 14:00:00', '2023-06-12 15:00:00',
               '2023-06-12 16:00:00', '2023-06-12 17:00:00',
               '2023-06-12 18:00:00', '2023-06-12 19:00:00',
               '2023-06-12 20:00:00', '2023-06-12 21:00:00',
               '2023-06-12 22:00:00', '2023-06-12 23:00:00',
               '2023-06-13 00:00:00', '2023-06-13 01:00:00',
               '2023-06-13 02:00:00', '2023-06-13 03:00:00',
               '2023-06-13 04:00:00', '2023-06-13 05:00:00',
               '2023-06-13 06:00:00', '2023-06-13 07:00:00',
               '2023-06-

### 3.8 Train Models


In [17]:
# Train SARIMA models per county on training subset
print("Training SARIMA models (one per county)...\n")
sarima_models = {}

# Notice we fixed SARIMA_ORDER for all counties, however, one can easily tune it over validation set for each county.
for loc in locations:
    print(f"Fitting SARIMA for {loc}...", end=" ")
    y_train = ds_train_sub.out.sel(location=loc).values.astype(float).flatten()
    model = safe_fit_sarima(y_train, order=SARIMA_ORDER)
    sarima_models[loc] = model
    if model is not None:
        print(f"✓ Success (AIC: {model.aic:.2f})")
    else:
        print("✗ Failed (will use zero baseline)")

print(f"\nSARIMA training complete!")

Training SARIMA models (one per county)...

Fitting SARIMA for 26001... ✓ Success (AIC: 22096.92)
Fitting SARIMA for 26003... ✓ Success (AIC: 20784.05)
Fitting SARIMA for 26005... ✓ Success (AIC: 20906.27)
Fitting SARIMA for 26007... ✓ Success (AIC: 10571.37)
Fitting SARIMA for 26009... ✓ Success (AIC: 21046.83)
Fitting SARIMA for 26011... ✓ Success (AIC: 14277.84)
Fitting SARIMA for 26013... ✓ Success (AIC: 16943.51)
Fitting SARIMA for 26015... ✓ Success (AIC: 17215.83)
Fitting SARIMA for 26017... ✓ Success (AIC: 15446.86)
Fitting SARIMA for 26019... ✓ Success (AIC: 18714.02)
Fitting SARIMA for 26021... ✓ Success (AIC: 20195.76)
Fitting SARIMA for 26023... ✓ Success (AIC: 22001.14)
Fitting SARIMA for 26025... ✓ Success (AIC: 22864.04)
Fitting SARIMA for 26027... ✓ Success (AIC: 19346.13)
Fitting SARIMA for 26029... ✓ Success (AIC: 17981.47)
Fitting SARIMA for 26031... ✓ Success (AIC: 17534.75)
Fitting SARIMA for 26033... ✓ Success (AIC: 20349.10)
Fitting SARIMA for 26035... ✓ Success 

### 3.9 Generate Predictions

In [18]:
# Generate predictions using consolidated functions
print("Generating predictions for validation set...\n")

# SARIMA predictions (same models for both horizons, just different forecast lengths)
print("1) SARIMA predictions...")
sarima_val_pred_24h_df = generate_sarima_predictions(sarima_models, locations, val_timestamps_24h)
sarima_val_pred_48h_df = generate_sarima_predictions(sarima_models, locations, val_timestamps_48h)
print(f"   24h: {len(sarima_val_pred_24h_df)} predictions")
print(f"   48h: {len(sarima_val_pred_48h_df)} predictions")

Generating predictions for validation set...

1) SARIMA predictions...
   24h: 1992 predictions
   48h: 3984 predictions


### 3.10 Evaluate Performance on Validation Set

In [19]:

# Helper function to evaluate per-county RMSE
def evaluate_per_county(truth, pred_df, locations):
    """Evaluate RMSE per county and return list of RMSEs"""
    rmses = []
    for i, loc in enumerate(locations):
        loc_str = str(loc)
        loc_pred = pred_df[pred_df['location'].astype(str) == loc_str]['pred'].values
        if len(loc_pred) == truth.shape[0]:
            county_rmse = rmse(truth[:, i], loc_pred)
            rmses.append(county_rmse)
        else:
            rmses.append(np.nan)
    return rmses

# Evaluate 24h horizon
print("="*70)
print("VALIDATION EVALUATION - 24H HORIZON")
print("="*70)

sarima_24h_rmses = evaluate_per_county(val_truth_24h, sarima_val_pred_24h_df, locations)
# seq2seq_24h_rmses = evaluate_per_county(val_truth_24h, seq2seq_val_pred_24h_df, locations)
zero_24h_rmses = [rmse(val_truth_24h[:, i], np.zeros(24)) for i in range(len(locations))]

sarima_24h_avg = np.nanmean(sarima_24h_rmses)
# seq2seq_24h_avg = np.nanmean(seq2seq_24h_rmses)
zero_24h_avg = np.nanmean(zero_24h_rmses)

print(f"\nAverage RMSE across counties:")
print(f"  SARIMA: {sarima_24h_avg:.4f}")
# print(f"  Seq2Seq: {seq2seq_24h_avg:.4f}")
print(f"  Zero Baseline: {zero_24h_avg:.4f}")

sarima_24h_imp = ((zero_24h_avg - sarima_24h_avg) / zero_24h_avg * 100)
# seq2seq_24h_imp = ((zero_24h_avg - seq2seq_24h_avg) / zero_24h_avg * 100)

# print(f"\nBest Model for 24h: ", end="")
# if sarima_24h_avg < seq2seq_24h_avg and sarima_24h_avg < zero_24h_avg:
#     print(f"✓ SARIMA ({sarima_24h_avg:.4f})")
# elif seq2seq_24h_avg < sarima_24h_avg and seq2seq_24h_avg < zero_24h_avg:
#     print(f"✓ Seq2Seq ({seq2seq_24h_avg:.4f})")
# else:
#     print(f"⚠ Zero Baseline is better ({zero_24h_avg:.4f})")

# Evaluate 48h horizon
print("\n" + "="*70)
print("VALIDATION EVALUATION - 48H HORIZON")
print("="*70)

sarima_48h_rmses = evaluate_per_county(val_truth_48h, sarima_val_pred_48h_df, locations)
# seq2seq_48h_rmses = evaluate_per_county(val_truth_48h, seq2seq_val_pred_48h_df, locations)
zero_48h_rmses = [rmse(val_truth_48h[:, i], np.zeros(48)) for i in range(len(locations))]

sarima_48h_avg = np.nanmean(sarima_48h_rmses)
# seq2seq_48h_avg = np.nanmean(seq2seq_48h_rmses)
zero_48h_avg = np.nanmean(zero_48h_rmses)

print(f"\nAverage RMSE across counties:")
print(f"  SARIMA: {sarima_48h_avg:.4f}")
# print(f"  Seq2Seq: {seq2seq_48h_avg:.4f}")
print(f"  Zero Baseline: {zero_48h_avg:.4f}")

sarima_48h_imp = ((zero_48h_avg - sarima_48h_avg) / zero_48h_avg * 100)
# seq2seq_48h_imp = ((zero_48h_avg - seq2seq_48h_avg) / zero_48h_avg * 100)

VALIDATION EVALUATION - 24H HORIZON

Average RMSE across counties:
  SARIMA: 89.7450
  Zero Baseline: 100.3053

VALIDATION EVALUATION - 48H HORIZON

Average RMSE across counties:
  SARIMA: 68.3606
  Zero Baseline: 75.3561


## 4. Tuned orders for top 10 worst counties

### 4.1 24H

In [82]:
import pandas as pd

# Create a DataFrame with locations and their corresponding RMSEs
rmse_df = pd.DataFrame({
    'location': locations,
    'rmse_24h': sarima_24h_rmses
})

# Sort by RMSE descending to find the worst performing counties
top_worst_counties = rmse_df.sort_values(by='rmse_24h', ascending=False).reset_index(drop=True)

print("Top 10 Counties with Highest Validation RMSE (24h Horizon):")
display(top_worst_counties.head(10))

Top 10 Counties with Highest Validation RMSE (24h Horizon):


,location,rmse_24h
0,26125,1626.676181
1,26163,1385.476389
2,26099,786.798246
3,26139,712.500698
4,26087,606.525192
5,26115,314.551063
6,26021,260.230040
7,26065,222.754602
8,26049,208.925794
9,26093,146.467487


In [83]:
import itertools
import warnings
warnings.filterwarnings('ignore')

# Extract the top 10 worst locations
top_10_locations = top_worst_counties['location'].head(10).tolist()

print("="*70)
print("Tuning SARIMA (p,d,q) for the Top 10 Worst Locations")
print("="*70)

# Define parameter range for p, d, q
p = q = range(0, 3)
d = [0, 1]
pdq_combinations = list(itertools.product(p, d, q))

best_orders_top10 = {}

for loc in top_10_locations:
    print(f"\nTuning for location {loc}...")
    y_train_loc = ds_train_sub.out.sel(location=loc).values.astype(float).flatten()

    best_aic = float("inf")
    best_order = None
    best_model_res = None

    for order in pdq_combinations:
        try:
            model = SARIMAX(y_train_loc, order=order, enforce_stationarity=False, enforce_invertibility=False)
            res = model.fit(disp=False)
            if res.aic < best_aic:
                best_aic = res.aic
                best_order = order
                best_model_res = res
        except Exception:
            continue

    if best_order is not None:
        print(f"  ✓ Best Order: {best_order} (AIC: {best_aic:.2f})")
        best_orders_top10[loc] = best_order

        # Update the global sarima_models dictionary with the tuned model
        sarima_models[str(loc)] = best_model_res
    else:
        print(f"  ✗ Failed to find a valid order for {loc}")

print("\nTuning complete for top 10 worst locations! Our sarima_models dictionary has been updated.")

Tuning SARIMA (p,d,q) for the Top 10 Worst Locations

Tuning for location 26125...
  ✓ Best Order: (2, 1, 2) (AIC: 27075.25)

Tuning for location 26163...
  ✓ Best Order: (2, 1, 2) (AIC: 26800.37)

Tuning for location 26099...
  ✓ Best Order: (2, 1, 2) (AIC: 25797.17)

Tuning for location 26139...
  ✓ Best Order: (2, 1, 2) (AIC: 21193.15)

Tuning for location 26087...
  ✓ Best Order: (1, 1, 2) (AIC: 18925.86)

Tuning for location 26115...
  ✓ Best Order: (2, 1, 2) (AIC: 21426.36)

Tuning for location 26021...
  ✓ Best Order: (2, 0, 2) (AIC: 20166.72)

Tuning for location 26065...
  ✓ Best Order: (2, 0, 2) (AIC: 22940.70)

Tuning for location 26049...
  ✓ Best Order: (2, 0, 2) (AIC: 22460.93)

Tuning for location 26093...
  ✓ Best Order: (1, 1, 2) (AIC: 21038.55)

Tuning complete for top 10 worst locations! Our sarima_models dictionary has been updated.


In [84]:
import pandas as pd

print("Evaluating Tuned vs Original SARIMA for Top 10 Worst Locations (24h Horizon)\n")

# Generate predictions for just the top 10 locations using the updated models
tuned_preds_top10_df = generate_sarima_predictions(
    sarima_models,
    top_10_locations,
    val_timestamps_24h
)

results = []

for loc in top_10_locations:
    # Get original RMSE
    loc_idx = locations.index(loc)
    original_rmse_val = sarima_24h_rmses[loc_idx]

    # Get new predictions
    loc_str = str(loc)
    loc_pred = tuned_preds_top10_df[tuned_preds_top10_df['location'].astype(str) == loc_str]['pred'].values

    # Get truth for this specific location
    truth_loc = val_truth_24h[:, loc_idx]

    # Calculate new RMSE
    if len(loc_pred) == len(truth_loc):
        new_rmse_val = rmse(truth_loc, loc_pred)
    else:
        new_rmse_val = np.nan

    improvement = original_rmse_val - new_rmse_val
    pct_improvement = (improvement / original_rmse_val * 100) if original_rmse_val > 0 else 0

    results.append({
        'Location': loc,
        'Original Order': SARIMA_ORDER,
        'Original RMSE': original_rmse_val,
        'Tuned Order': best_orders_top10.get(loc, "Failed"),
        'Tuned RMSE': new_rmse_val,
        'Improvement (%)': pct_improvement
    })

comparison_df = pd.DataFrame(results)
display(comparison_df.round(4))

Evaluating Tuned vs Original SARIMA for Top 10 Worst Locations (24h Horizon)



,Location,Original Order,Original RMSE,Tuned Order,Tuned RMSE,Improvement (%)
0,26125,"(1, 0, 1)",1626.6762,"(2, 1, 2)",1319.0243,18.9129
1,26163,"(1, 0, 1)",1385.4764,"(2, 1, 2)",1077.2712,22.2454
2,26099,"(1, 0, 1)",786.7982,"(2, 1, 2)",673.7939,14.3626
3,26139,"(1, 0, 1)",712.5007,"(2, 1, 2)",711.5255,0.1369
4,26087,"(1, 0, 1)",606.5252,"(1, 1, 2)",799.9246,-31.8865
5,26115,"(1, 0, 1)",314.5511,"(2, 1, 2)",313.0085,0.4904
6,26021,"(1, 0, 1)",260.2300,"(2, 0, 2)",259.8964,0.1282
7,26065,"(1, 0, 1)",222.7546,"(2, 0, 2)",222.7545,0.0001
8,26049,"(1, 0, 1)",208.9258,"(2, 0, 2)",207.6372,0.6168
9,26093,"(1, 0, 1)",146.4675,"(1, 1, 2)",145.9607,0.3460


In [85]:
print("="*70)
print("OVERALL VALIDATION EVALUATION - 24H HORIZON (AFTER TUNING TOP 10)")
print("="*70)

# Generate predictions for all locations using the updated sarima_models
tuned_all_preds_24h_df = generate_sarima_predictions(sarima_models, locations, val_timestamps_24h)

# Evaluate per county
tuned_all_rmses = evaluate_per_county(val_truth_24h, tuned_all_preds_24h_df, locations)

# Calculate new average RMSE
tuned_all_avg = np.nanmean(tuned_all_rmses)

print(f"Original Overall Average RMSE: {sarima_24h_avg:.4f}")

print(f"New Overall Average RMSE (with Top 10 Tuned): {tuned_all_avg:.4f}")

if tuned_all_avg < sarima_24h_avg:
    improvement = (sarima_24h_avg - tuned_all_avg) / sarima_24h_avg * 100
    print(f"\n✓ Tuning the top 10 locations improved the overall RMSE by {improvement:.2f}%")
elif tuned_all_avg > sarima_24h_avg:
    degradation = (tuned_all_avg - sarima_24h_avg) / sarima_24h_avg * 100
    print(f"\n✗ Tuning the top 10 locations worsened the overall RMSE by {degradation:.2f}%")
else:
    print("\n- No change in overall RMSE.")

OVERALL VALIDATION EVALUATION - 24H HORIZON (AFTER TUNING TOP 10)
Original Overall Average RMSE: 89.7450
New Overall Average RMSE (with Top 10 Tuned): 83.2376

✓ Tuning the top 10 locations improved the overall RMSE by 7.25%


In [86]:
import pandas as pd
import numpy as np

print("="*70)
print("FALLBACK STRATEGY: Default (1,0,1) vs Tuned Order for Top 10")
print("="*70)

# We will store the final approved models here
final_validated_models = sarima_models.copy()
validation_results = []

for loc in top_10_locations:
    loc_str = str(loc)
    y_train_loc = ds_train_sub.out.sel(location=loc).values.astype(float).flatten()

    # 1. Base Model (1, 0, 1)
    base_model = safe_fit_sarima(y_train_loc, order=SARIMA_ORDER)

    # 2. Tuned Model (currently stored in sarima_models from previous tuning)
    tuned_model = sarima_models.get(loc_str)
    tuned_order = best_orders_top10.get(loc, "Failed")

    # Extract truth for this location
    loc_idx = locations.index(loc)
    truth_loc = val_truth_24h[:, loc_idx]

    # Get base prediction & RMSE
    if base_model is not None:
        base_pred = generate_sarima_predictions({loc_str: base_model}, [loc], val_timestamps_24h)
        base_rmse = rmse(truth_loc, base_pred['pred'].values)
    else:
        base_rmse = float('inf')

    # Get tuned prediction & RMSE
    if tuned_model is not None:
        tuned_pred = generate_sarima_predictions({loc_str: tuned_model}, [loc], val_timestamps_24h)
        tuned_rmse = rmse(truth_loc, tuned_pred['pred'].values)
    else:
        tuned_rmse = float('inf')

    # Compare & Pick Winner
    if tuned_rmse < base_rmse:
        print(f"Location {loc}: Keeping Tuned {tuned_order} (RMSE: {tuned_rmse:.4f} < Base: {base_rmse:.4f})")
        final_validated_models[loc_str] = tuned_model
        winner = "Tuned"
        final_rmse = tuned_rmse
    else:
        print(f"Location {loc}: Reverting to Base {SARIMA_ORDER} (RMSE: {base_rmse:.4f} <= Tuned: {tuned_rmse:.4f})")
        final_validated_models[loc_str] = base_model
        winner = "Base"
        final_rmse = base_rmse

    validation_results.append({
        'Location': loc,
        'Base Order': SARIMA_ORDER,
        'Base RMSE': base_rmse,
        'Tuned Order': tuned_order,
        'Tuned RMSE': tuned_rmse,
        'Winner': winner,
        'Final RMSE': final_rmse
    })

print("\nFallback selection complete!")
display(pd.DataFrame(validation_results).round(4))

FALLBACK STRATEGY: Default (1,0,1) vs Tuned Order for Top 10
Location 26125: Keeping Tuned (2, 1, 2) (RMSE: 1319.0243 < Base: 1626.6762)
Location 26163: Keeping Tuned (2, 1, 2) (RMSE: 1077.2712 < Base: 1385.4764)
Location 26099: Keeping Tuned (2, 1, 2) (RMSE: 673.7939 < Base: 786.7982)
Location 26139: Keeping Tuned (2, 1, 2) (RMSE: 711.5255 < Base: 712.5007)
Location 26087: Reverting to Base (1, 0, 1) (RMSE: 606.5252 <= Tuned: 799.9246)
Location 26115: Keeping Tuned (2, 1, 2) (RMSE: 313.0085 < Base: 314.5511)
Location 26021: Keeping Tuned (2, 0, 2) (RMSE: 259.8964 < Base: 260.2300)
Location 26065: Keeping Tuned (2, 0, 2) (RMSE: 222.7545 < Base: 222.7546)
Location 26049: Keeping Tuned (2, 0, 2) (RMSE: 207.6372 < Base: 208.9258)
Location 26093: Keeping Tuned (1, 1, 2) (RMSE: 145.9607 < Base: 146.4675)

Fallback selection complete!


,Location,Base Order,Base RMSE,Tuned Order,Tuned RMSE,Winner,Final RMSE
0,26125,"(1, 0, 1)",1626.6762,"(2, 1, 2)",1319.0243,Tuned,1319.0243
1,26163,"(1, 0, 1)",1385.4764,"(2, 1, 2)",1077.2712,Tuned,1077.2712
2,26099,"(1, 0, 1)",786.7982,"(2, 1, 2)",673.7939,Tuned,673.7939
3,26139,"(1, 0, 1)",712.5007,"(2, 1, 2)",711.5255,Tuned,711.5255
4,26087,"(1, 0, 1)",606.5252,"(1, 1, 2)",799.9246,Base,606.5252
5,26115,"(1, 0, 1)",314.5511,"(2, 1, 2)",313.0085,Tuned,313.0085
6,26021,"(1, 0, 1)",260.2300,"(2, 0, 2)",259.8964,Tuned,259.8964
7,26065,"(1, 0, 1)",222.7546,"(2, 0, 2)",222.7545,Tuned,222.7545
8,26049,"(1, 0, 1)",208.9258,"(2, 0, 2)",207.6372,Tuned,207.6372
9,26093,"(1, 0, 1)",146.4675,"(1, 1, 2)",145.9607,Tuned,145.9607


In [87]:
print("="*70)
print("OVERALL VALIDATION EVALUATION - 24H HORIZON (WITH FALLBACK)")
print("="*70)

# Generate predictions for all locations using the final validated models
validated_all_preds_24h_df = generate_sarima_predictions(final_validated_models, locations, val_timestamps_24h)

# Evaluate per county
validated_all_rmses = evaluate_per_county(val_truth_24h, validated_all_preds_24h_df, locations)

# Calculate new average RMSE
validated_all_avg = np.nanmean(validated_all_rmses)

print(f"Original Overall Average RMSE (Base models only): {sarima_24h_avg:.4f}")
print(f"New Overall Average RMSE (with Fallback Strategy): {validated_all_avg:.4f}")

if validated_all_avg < sarima_24h_avg:
    improvement = (sarima_24h_avg - validated_all_avg) / sarima_24h_avg * 100
    print(f"\n✓ Fallback strategy improved the overall RMSE by {improvement:.2f}%")
elif validated_all_avg > sarima_24h_avg:
    degradation = (validated_all_avg - sarima_24h_avg) / sarima_24h_avg * 100
    print(f"\n✗ Fallback strategy worsened the overall RMSE by {degradation:.2f}%")
else:
    print("\n- No change in overall RMSE.")

# Update sarima_models globally to reflect these fallback choices for downstream tasks
sarima_models = final_validated_models

OVERALL VALIDATION EVALUATION - 24H HORIZON (WITH FALLBACK)
Original Overall Average RMSE (Base models only): 89.7450
New Overall Average RMSE (with Fallback Strategy): 80.9075

✓ Fallback strategy improved the overall RMSE by 9.85%


In [88]:
import joblib
import pandas as pd
import os

# --- Step 1: Define save directory ---
save_dir = os.path.join(RESULTS_DIR, "sarima_models_24H_all_counties")
os.makedirs(save_dir, exist_ok=True)

print(f"Saving all SARIMA models to: {save_dir}")

saved_count = 0
# Iterate through all locations and save their respective models
for loc in locations:
    model_to_save = sarima_models.get(loc) # Get model from the global sarima_models dictionary
    if model_to_save:
        # Filename updated as requested
        filename = f"sarima_model_24H_{loc}.joblib"
        filepath = os.path.join(save_dir, filename)
        joblib.dump(model_to_save, filepath)
        print(f"  Saved {filename}")
        saved_count += 1
    else:
        print(f"  Warning: No model found for location {loc} in `sarima_models`. Skipping save.")

print(f"\nSuccessfully saved {saved_count} models.")
print("To load a model, you can use: `loaded_model = joblib.load('path/to/model.joblib')`")

Saving all SARIMA models to: drive/MyDrive/Colab Notebooks/MLPS_proj/results/sarima_models_24H_all_counties
  Saved sarima_model_24H_26001.joblib
  Saved sarima_model_24H_26003.joblib
  Saved sarima_model_24H_26005.joblib
  Saved sarima_model_24H_26007.joblib
  Saved sarima_model_24H_26009.joblib
  Saved sarima_model_24H_26011.joblib
  Saved sarima_model_24H_26013.joblib
  Saved sarima_model_24H_26015.joblib
  Saved sarima_model_24H_26017.joblib
  Saved sarima_model_24H_26019.joblib
  Saved sarima_model_24H_26021.joblib
  Saved sarima_model_24H_26023.joblib
  Saved sarima_model_24H_26025.joblib
  Saved sarima_model_24H_26027.joblib
  Saved sarima_model_24H_26029.joblib
  Saved sarima_model_24H_26031.joblib
  Saved sarima_model_24H_26033.joblib
  Saved sarima_model_24H_26035.joblib
  Saved sarima_model_24H_26037.joblib
  Saved sarima_model_24H_26039.joblib
  Saved sarima_model_24H_26041.joblib
  Saved sarima_model_24H_26043.joblib
  Saved sarima_model_24H_26045.joblib
  Saved sarima_mod

### 48H

In [89]:
import pandas as pd

# Create a DataFrame with locations and their corresponding RMSEs
rmse_df = pd.DataFrame({
    'location': locations,
    'rmse_48h': sarima_48h_rmses
})

# Sort by RMSE descending to find the worst performing counties
top_worst_counties = rmse_df.sort_values(by='rmse_48h', ascending=False).reset_index(drop=True)

print("Top 10 Counties with Highest Validation RMSE (48h Horizon):")
display(top_worst_counties.head(10))

Top 10 Counties with Highest Validation RMSE (48h Horizon):


,location,rmse_48h
0,26125,1154.226711
1,26163,992.208309
2,26099,569.362720
3,26139,516.697623
4,26087,431.068537
5,26115,222.456527
6,26021,188.639275
7,26065,163.660326
8,26049,162.321998
9,26081,137.590448


In [91]:
import itertools
import warnings
warnings.filterwarnings('ignore')

# Extract the top 10 worst locations
top_10_locations = top_worst_counties['location'].head(10).tolist()

print("="*70)
print("Tuning SARIMA (p,d,q) for the Top 10 Worst Locations")
print("="*70)

# Define parameter range for p, d, q
p = q = range(0, 3)
d = [0, 1]
pdq_combinations = list(itertools.product(p, d, q))

best_orders_top10 = {}
sarima_models_48h_top10 = {} # New variable to store tuned models for 48h top 10

for loc in top_10_locations:
    print(f"\nTuning for location {loc}...")
    y_train_loc = ds_train_sub.out.sel(location=loc).values.astype(float).flatten()

    best_aic = float("inf")
    best_order = None
    best_model_res = None

    for order in pdq_combinations:
        try:
            model = SARIMAX(y_train_loc, order=order, enforce_stationarity=False, enforce_invertibility=False)
            res = model.fit(disp=False)
            if res.aic < best_aic:
                best_aic = res.aic
                best_order = order
                best_model_res = res
        except Exception:
            continue

    if best_order is not None:
        print(f"  ✓ Best Order: {best_order} (AIC: {best_aic:.2f})")
        best_orders_top10[loc] = best_order

        # Update the new separate dictionary with the tuned model
        sarima_models_48h_top10[str(loc)] = best_model_res
    else:
        print(f"  ✗ Failed to find a valid order for {loc}")

print("\nTuning complete for top 10 worst locations! Models are stored in `sarima_models_48h_top10`.")

Tuning SARIMA (p,d,q) for the Top 10 Worst Locations

Tuning for location 26125...
  ✓ Best Order: (2, 1, 2) (AIC: 27075.25)

Tuning for location 26163...
  ✓ Best Order: (2, 1, 2) (AIC: 26800.37)

Tuning for location 26099...
  ✓ Best Order: (2, 1, 2) (AIC: 25797.17)

Tuning for location 26139...
  ✓ Best Order: (2, 1, 2) (AIC: 21193.15)

Tuning for location 26087...
  ✓ Best Order: (1, 1, 2) (AIC: 18925.86)

Tuning for location 26115...
  ✓ Best Order: (2, 1, 2) (AIC: 21426.36)

Tuning for location 26021...
  ✓ Best Order: (2, 0, 2) (AIC: 20166.72)

Tuning for location 26065...
  ✓ Best Order: (2, 0, 2) (AIC: 22940.70)

Tuning for location 26049...
  ✓ Best Order: (2, 0, 2) (AIC: 22460.93)

Tuning for location 26081...
  ✓ Best Order: (2, 0, 2) (AIC: 23552.52)

Tuning complete for top 10 worst locations! Models are stored in `sarima_models_48h_top10`.


In [92]:
import pandas as pd

print("Evaluating Tuned vs Original SARIMA for Top 10 Worst Locations (48h Horizon)\n")

# Generate predictions for just the top 10 locations using the updated models
tuned_preds_top10_df = generate_sarima_predictions(
    sarima_models_48h_top10,
    top_10_locations,
    val_timestamps_48h
)

results = []

for loc in top_10_locations:
    # Get original RMSE
    loc_idx = locations.index(loc)
    original_rmse_val = sarima_48h_rmses[loc_idx]

    # Get new predictions
    loc_str = str(loc)
    loc_pred = tuned_preds_top10_df[tuned_preds_top10_df['location'].astype(str) == loc_str]['pred'].values

    # Get truth for this specific location
    truth_loc = val_truth_48h[:, loc_idx]

    # Calculate new RMSE
    if len(loc_pred) == len(truth_loc):
        new_rmse_val = rmse(truth_loc, loc_pred)
    else:
        new_rmse_val = np.nan

    improvement = original_rmse_val - new_rmse_val
    pct_improvement = (improvement / original_rmse_val * 100) if original_rmse_val > 0 else 0

    results.append({
        'Location': loc,
        'Original Order': SARIMA_ORDER,
        'Original RMSE': original_rmse_val,
        'Tuned Order': best_orders_top10.get(loc, "Failed"),
        'Tuned RMSE': new_rmse_val,
        'Improvement (%)': pct_improvement
    })

comparison_df = pd.DataFrame(results)
display(comparison_df.round(4))

Evaluating Tuned vs Original SARIMA for Top 10 Worst Locations (48h Horizon)



,Location,Original Order,Original RMSE,Tuned Order,Tuned RMSE,Improvement (%)
0,26125,"(1, 0, 1)",1154.2267,"(2, 1, 2)",978.3910,15.2341
1,26163,"(1, 0, 1)",992.2083,"(2, 1, 2)",776.2614,21.7643
2,26099,"(1, 0, 1)",569.3627,"(2, 1, 2)",480.5569,15.5974
3,26139,"(1, 0, 1)",516.6976,"(2, 1, 2)",514.6647,0.3934
4,26087,"(1, 0, 1)",431.0685,"(1, 1, 2)",787.3837,-82.6586
5,26115,"(1, 0, 1)",222.4565,"(2, 1, 2)",221.3370,0.5032
6,26021,"(1, 0, 1)",188.6393,"(2, 0, 2)",188.4052,0.1241
7,26065,"(1, 0, 1)",163.6603,"(2, 0, 2)",163.6602,0.0001
8,26049,"(1, 0, 1)",162.3220,"(2, 0, 2)",161.4931,0.5106
9,26081,"(1, 0, 1)",137.5904,"(2, 0, 2)",134.4015,2.3177


In [94]:
import pandas as pd
import numpy as np

print("="*70)
print("FALLBACK STRATEGY: Default (1,0,1) vs Tuned Order for Top 10 (48H)")
print("="*70)

# We will store the final approved models here
# For 48h, we use a separate dictionary to store models, as per previous decision
final_validated_models_48h = sarima_models.copy() # Start with all default models
validation_results_48h = []

for loc in top_10_locations:
    loc_str = str(loc)
    y_train_loc = ds_train_sub.out.sel(location=loc).values.astype(float).flatten()

    # 1. Base Model (1, 0, 1)
    base_model = safe_fit_sarima(y_train_loc, order=SARIMA_ORDER)

    # 2. Tuned Model (currently stored in sarima_models_48h_top10 for this horizon)
    tuned_model = sarima_models_48h_top10.get(loc_str)
    tuned_order = best_orders_top10.get(loc, "Failed")

    # Extract truth for this location (48h)
    loc_idx = locations.index(loc)
    truth_loc = val_truth_48h[:, loc_idx]

    # Get base prediction & RMSE (48h)
    if base_model is not None:
        base_pred = generate_sarima_predictions({loc_str: base_model}, [loc], val_timestamps_48h)
        base_rmse = rmse(truth_loc, base_pred['pred'].values)
    else:
        base_rmse = float('inf')

    # Get tuned prediction & RMSE (48h)
    if tuned_model is not None:
        tuned_pred = generate_sarima_predictions({loc_str: tuned_model}, [loc], val_timestamps_48h)
        tuned_rmse = rmse(truth_loc, tuned_pred['pred'].values)
    else:
        tuned_rmse = float('inf')

    # Compare & Pick Winner
    if tuned_rmse < base_rmse:
        print(f"Location {loc}: Keeping Tuned {tuned_order} (RMSE: {tuned_rmse:.4f} < Base: {base_rmse:.4f})")
        final_validated_models_48h[loc_str] = tuned_model
        winner = "Tuned"
        final_rmse = tuned_rmse
    else:
        print(f"Location {loc}: Reverting to Base {SARIMA_ORDER} (RMSE: {base_rmse:.4f} <= Tuned: {tuned_rmse:.4f})")
        final_validated_models_48h[loc_str] = base_model
        winner = "Base"
        final_rmse = base_rmse

    validation_results_48h.append({
        'Location': loc,
        'Base Order': SARIMA_ORDER,
        'Base RMSE': base_rmse,
        'Tuned Order': tuned_order,
        'Tuned RMSE': tuned_rmse,
        'Winner': winner,
        'Final RMSE': final_rmse
    })

print("\nFallback selection complete for 48H!")
display(pd.DataFrame(validation_results_48h).round(4))

FALLBACK STRATEGY: Default (1,0,1) vs Tuned Order for Top 10 (48H)
Location 26125: Keeping Tuned (2, 1, 2) (RMSE: 978.3910 < Base: 1154.2267)
Location 26163: Keeping Tuned (2, 1, 2) (RMSE: 776.2614 < Base: 992.2083)
Location 26099: Keeping Tuned (2, 1, 2) (RMSE: 480.5569 < Base: 569.3627)
Location 26139: Keeping Tuned (2, 1, 2) (RMSE: 514.6647 < Base: 516.6976)
Location 26087: Reverting to Base (1, 0, 1) (RMSE: 431.0685 <= Tuned: 787.3837)
Location 26115: Keeping Tuned (2, 1, 2) (RMSE: 221.3370 < Base: 222.4565)
Location 26021: Keeping Tuned (2, 0, 2) (RMSE: 188.4052 < Base: 188.6393)
Location 26065: Keeping Tuned (2, 0, 2) (RMSE: 163.6602 < Base: 163.6603)
Location 26049: Keeping Tuned (2, 0, 2) (RMSE: 161.4931 < Base: 162.3220)
Location 26081: Keeping Tuned (2, 0, 2) (RMSE: 134.4015 < Base: 137.5904)

Fallback selection complete for 48H!


,Location,Base Order,Base RMSE,Tuned Order,Tuned RMSE,Winner,Final RMSE
0,26125,"(1, 0, 1)",1154.2267,"(2, 1, 2)",978.3910,Tuned,978.3910
1,26163,"(1, 0, 1)",992.2083,"(2, 1, 2)",776.2614,Tuned,776.2614
2,26099,"(1, 0, 1)",569.3627,"(2, 1, 2)",480.5569,Tuned,480.5569
3,26139,"(1, 0, 1)",516.6976,"(2, 1, 2)",514.6647,Tuned,514.6647
4,26087,"(1, 0, 1)",431.0685,"(1, 1, 2)",787.3837,Base,431.0685
5,26115,"(1, 0, 1)",222.4565,"(2, 1, 2)",221.3370,Tuned,221.3370
6,26021,"(1, 0, 1)",188.6393,"(2, 0, 2)",188.4052,Tuned,188.4052
7,26065,"(1, 0, 1)",163.6603,"(2, 0, 2)",163.6602,Tuned,163.6602
8,26049,"(1, 0, 1)",162.3220,"(2, 0, 2)",161.4931,Tuned,161.4931
9,26081,"(1, 0, 1)",137.5904,"(2, 0, 2)",134.4015,Tuned,134.4015


In [96]:
print("="*70)
print("OVERALL VALIDATION EVALUATION - 48H HORIZON (WITH FALLBACK)")
print("="*70)

# Generate predictions for all locations using the final validated models for 48H
validated_all_preds_48h_df = generate_sarima_predictions(final_validated_models_48h, locations, val_timestamps_48h)

# Evaluate per county for 48H
validated_all_rmses_48h = evaluate_per_county(val_truth_48h, validated_all_preds_48h_df, locations)

# Calculate new average RMSE for 48H
validated_all_avg_48h = np.nanmean(validated_all_rmses_48h)

print(f"Original Overall Average RMSE (Base models only, 48H): {sarima_48h_avg:.4f}")
print(f"New Overall Average RMSE (with Fallback Strategy, 48H): {validated_all_avg_48h:.4f}")

if validated_all_avg_48h < sarima_48h_avg:
    improvement = (sarima_48h_avg - validated_all_avg_48h) / sarima_48h_avg * 100
    print(f"\n✓ Fallback strategy improved the overall RMSE for 48H by {improvement:.2f}%")
elif validated_all_avg_48h > sarima_48h_avg:
    degradation = (validated_all_avg_48h - sarima_48h_avg) / sarima_48h_avg * 100
    print(f"\n✗ Fallback strategy worsened the overall RMSE for 48H by {degradation:.2f}%")
else:
    print("\n- No change in overall RMSE for 48H.")

# Update sarima_models globally to reflect these fallback choices for 48H for downstream tasks
sarima_models = final_validated_models_48h

OVERALL VALIDATION EVALUATION - 48H HORIZON (WITH FALLBACK)
Original Overall Average RMSE (Base models only, 48H): 68.3606
New Overall Average RMSE (with Fallback Strategy, 48H): 62.5780

✓ Fallback strategy improved the overall RMSE for 48H by 8.46%


In [97]:
import joblib
import pandas as pd
import os

# --- Step 1: Define save directory ---
save_dir = os.path.join(RESULTS_DIR, "sarima_models_48H_all_counties")
os.makedirs(save_dir, exist_ok=True)

print(f"Saving all SARIMA models to: {save_dir}")

saved_count = 0
# Iterate through all locations and save their respective models
for loc in locations:
    model_to_save = sarima_models.get(loc) # Get model from the global sarima_models dictionary
    if model_to_save:
        # Filename updated as requested
        filename = f"sarima_model_48H_{loc}.joblib"
        filepath = os.path.join(save_dir, filename)
        joblib.dump(model_to_save, filepath)
        print(f"  Saved {filename}")
        saved_count += 1
    else:
        print(f"  Warning: No model found for location {loc} in `sarima_models`. Skipping save.")

print(f"\nSuccessfully saved {saved_count} models.")
print("To load a model, you can use: `loaded_model = joblib.load('path/to/model.joblib')`")

Saving all SARIMA models to: drive/MyDrive/Colab Notebooks/MLPS_proj/results/sarima_models_48H_all_counties
  Saved sarima_model_48H_26001.joblib
  Saved sarima_model_48H_26003.joblib
  Saved sarima_model_48H_26005.joblib
  Saved sarima_model_48H_26007.joblib
  Saved sarima_model_48H_26009.joblib
  Saved sarima_model_48H_26011.joblib
  Saved sarima_model_48H_26013.joblib
  Saved sarima_model_48H_26015.joblib
  Saved sarima_model_48H_26017.joblib
  Saved sarima_model_48H_26019.joblib
  Saved sarima_model_48H_26021.joblib
  Saved sarima_model_48H_26023.joblib
  Saved sarima_model_48H_26025.joblib
  Saved sarima_model_48H_26027.joblib
  Saved sarima_model_48H_26029.joblib
  Saved sarima_model_48H_26031.joblib
  Saved sarima_model_48H_26033.joblib
  Saved sarima_model_48H_26035.joblib
  Saved sarima_model_48H_26037.joblib
  Saved sarima_model_48H_26039.joblib
  Saved sarima_model_48H_26041.joblib
  Saved sarima_model_48H_26043.joblib
  Saved sarima_model_48H_26045.joblib
  Saved sarima_mod

## 5. Tuned orders for top 20 worst counties

### 5.1 24H

In [33]:
import pandas as pd

# Create a DataFrame with locations and their corresponding RMSEs
rmse_df = pd.DataFrame({
    'location': locations,
    'rmse_24h': sarima_24h_rmses
})

# Sort by RMSE descending to find the worst performing counties
top_worst_counties = rmse_df.sort_values(by='rmse_24h', ascending=False).reset_index(drop=True)

print("Top 20 Counties with Highest Validation RMSE (24h Horizon):")
display(top_worst_counties.head(20))

Top 20 Counties with Highest Validation RMSE (24h Horizon):


,location,rmse_24h
0,26125,1626.676181
1,26163,1385.476389
2,26099,786.798246
3,26139,712.500698
4,26087,606.525192
5,26115,314.551063
6,26021,260.230040
7,26065,222.754602
8,26049,208.925794
9,26093,146.467487


In [34]:
import itertools
import warnings
warnings.filterwarnings('ignore')

# Extract the top 20 worst locations
top_20_locations = top_worst_counties['location'].head(20).tolist()

print("="*70)
print("Tuning SARIMA (p,d,q) for the Top 20 Worst Locations")
print("="*70)

# Define parameter range for p, d, q
p = q = range(0, 3)
d = [0, 1]
pdq_combinations = list(itertools.product(p, d, q))

best_orders_top20 = {}

for loc in top_20_locations:
    print(f"\nTuning for location {loc}...")
    y_train_loc = ds_train_sub.out.sel(location=loc).values.astype(float).flatten()

    best_aic = float("inf")
    best_order = None
    best_model_res = None

    for order in pdq_combinations:
        try:
            model = SARIMAX(y_train_loc, order=order, enforce_stationarity=False, enforce_invertibility=False)
            res = model.fit(disp=False)
            if res.aic < best_aic:
                best_aic = res.aic
                best_order = order
                best_model_res = res
        except Exception:
            continue

    if best_order is not None:
        print(f"  ✓ Best Order: {best_order} (AIC: {best_aic:.2f})")
        best_orders_top20[loc] = best_order

        # Update the global sarima_models dictionary with the tuned model
        sarima_models[str(loc)] = best_model_res
    else:
        print(f"  ✗ Failed to find a valid order for {loc}")

print("\nTuning complete for top 20 worst locations! Our sarima_models dictionary has been updated.")

Tuning SARIMA (p,d,q) for the Top 20 Worst Locations

Tuning for location 26125...
  ✓ Best Order: (2, 1, 2) (AIC: 27075.25)

Tuning for location 26163...
  ✓ Best Order: (2, 1, 2) (AIC: 26800.37)

Tuning for location 26099...
  ✓ Best Order: (2, 1, 2) (AIC: 25797.17)

Tuning for location 26139...
  ✓ Best Order: (2, 1, 2) (AIC: 21193.15)

Tuning for location 26087...
  ✓ Best Order: (1, 1, 2) (AIC: 18925.86)

Tuning for location 26115...
  ✓ Best Order: (2, 1, 2) (AIC: 21426.36)

Tuning for location 26021...
  ✓ Best Order: (2, 0, 2) (AIC: 20166.72)

Tuning for location 26065...
  ✓ Best Order: (2, 0, 2) (AIC: 22940.70)

Tuning for location 26049...
  ✓ Best Order: (2, 0, 2) (AIC: 22460.93)

Tuning for location 26093...
  ✓ Best Order: (1, 1, 2) (AIC: 21038.55)

Tuning for location 26073...
  ✓ Best Order: (1, 0, 2) (AIC: 19973.21)

Tuning for location 26117...
  ✓ Best Order: (1, 1, 2) (AIC: 16002.66)

Tuning for location 26081...
  ✓ Best Order: (2, 0, 2) (AIC: 23552.52)

Tuning for

In [35]:
import pandas as pd

print("Evaluating Tuned vs Original SARIMA for Top 20 Worst Locations (24h Horizon)\n")

# Generate predictions for just the top 20 locations using the updated models
tuned_preds_top20_df = generate_sarima_predictions(
    sarima_models,
    top_20_locations,
    val_timestamps_24h
)

results = []

for loc in top_20_locations:
    # Get original RMSE
    loc_idx = locations.index(loc)
    original_rmse_val = sarima_24h_rmses[loc_idx]

    # Get new predictions
    loc_str = str(loc)
    loc_pred = tuned_preds_top20_df[tuned_preds_top20_df['location'].astype(str) == loc_str]['pred'].values

    # Get truth for this specific location
    truth_loc = val_truth_24h[:, loc_idx]

    # Calculate new RMSE
    if len(loc_pred) == len(truth_loc):
        new_rmse_val = rmse(truth_loc, loc_pred)
    else:
        new_rmse_val = np.nan

    improvement = original_rmse_val - new_rmse_val
    pct_improvement = (improvement / original_rmse_val * 100) if original_rmse_val > 0 else 0

    results.append({
        'Location': loc,
        'Original Order': SARIMA_ORDER,
        'Original RMSE': original_rmse_val,
        'Tuned Order': best_orders_top20.get(loc, "Failed"),
        'Tuned RMSE': new_rmse_val,
        'Improvement (%)': pct_improvement
    })

comparison_df = pd.DataFrame(results)
display(comparison_df.round(4))

Evaluating Tuned vs Original SARIMA for Top 20 Worst Locations (24h Horizon)



,Location,Original Order,Original RMSE,Tuned Order,Tuned RMSE,Improvement (%)
0,26125,"(1, 0, 1)",1626.6762,"(2, 1, 2)",1319.0243,18.9129
1,26163,"(1, 0, 1)",1385.4764,"(2, 1, 2)",1077.2712,22.2454
2,26099,"(1, 0, 1)",786.7982,"(2, 1, 2)",673.7939,14.3626
3,26139,"(1, 0, 1)",712.5007,"(2, 1, 2)",711.5255,0.1369
4,26087,"(1, 0, 1)",606.5252,"(1, 1, 2)",799.9246,-31.8865
5,26115,"(1, 0, 1)",314.5511,"(2, 1, 2)",313.0085,0.4904
6,26021,"(1, 0, 1)",260.2300,"(2, 0, 2)",259.8964,0.1282
7,26065,"(1, 0, 1)",222.7546,"(2, 0, 2)",222.7545,0.0001
8,26049,"(1, 0, 1)",208.9258,"(2, 0, 2)",207.6372,0.6168
9,26093,"(1, 0, 1)",146.4675,"(1, 1, 2)",145.9607,0.3460


In [36]:
print("="*70)
print("OVERALL VALIDATION EVALUATION - 24H HORIZON (AFTER TUNING TOP 20)")
print("="*70)

# Generate predictions for all locations using the updated sarima_models
tuned_all_preds_24h_df = generate_sarima_predictions(sarima_models, locations, val_timestamps_24h)

# Evaluate per county
tuned_all_rmses = evaluate_per_county(val_truth_24h, tuned_all_preds_24h_df, locations)

# Calculate new average RMSE
tuned_all_avg = np.nanmean(tuned_all_rmses)

print(f"Original Overall Average RMSE: {sarima_24h_avg:.4f}")

print(f"New Overall Average RMSE (with Top 20 Tuned): {tuned_all_avg:.4f}")

if tuned_all_avg < sarima_24h_avg:
    improvement = (sarima_24h_avg - tuned_all_avg) / sarima_24h_avg * 100
    print(f"\n✓ Tuning the top 20 locations improved the overall RMSE by {improvement:.2f}%")
elif tuned_all_avg > sarima_24h_avg:
    degradation = (tuned_all_avg - sarima_24h_avg) / sarima_24h_avg * 100
    print(f"\n✗ Tuning the top 20 locations worsened the overall RMSE by {degradation:.2f}%")
else:
    print("\n- No change in overall RMSE.")

OVERALL VALIDATION EVALUATION - 24H HORIZON (AFTER TUNING TOP 20)
Original Overall Average RMSE: 89.7450
New Overall Average RMSE (with Top 20 Tuned): 83.1939

✓ Tuning the top 20 locations improved the overall RMSE by 7.30%


In [37]:
import pandas as pd
import numpy as np

print("="*70)
print("FALLBACK STRATEGY: Default (1,0,1) vs Tuned Order for Top 20")
print("="*70)

# We will store the final approved models here
final_validated_models = sarima_models.copy()
validation_results = []

for loc in top_20_locations:
    loc_str = str(loc)
    y_train_loc = ds_train_sub.out.sel(location=loc).values.astype(float).flatten()

    # 1. Base Model (1, 0, 1)
    base_model = safe_fit_sarima(y_train_loc, order=SARIMA_ORDER)

    # 2. Tuned Model (currently stored in sarima_models from previous tuning)
    tuned_model = sarima_models.get(loc_str)
    tuned_order = best_orders_top20.get(loc, "Failed")

    # Extract truth for this location
    loc_idx = locations.index(loc)
    truth_loc = val_truth_24h[:, loc_idx]

    # Get base prediction & RMSE
    if base_model is not None:
        base_pred = generate_sarima_predictions({loc_str: base_model}, [loc], val_timestamps_24h)
        base_rmse = rmse(truth_loc, base_pred['pred'].values)
    else:
        base_rmse = float('inf')

    # Get tuned prediction & RMSE
    if tuned_model is not None:
        tuned_pred = generate_sarima_predictions({loc_str: tuned_model}, [loc], val_timestamps_24h)
        tuned_rmse = rmse(truth_loc, tuned_pred['pred'].values)
    else:
        tuned_rmse = float('inf')

    # Compare & Pick Winner
    if tuned_rmse < base_rmse:
        print(f"Location {loc}: Keeping Tuned {tuned_order} (RMSE: {tuned_rmse:.4f} < Base: {base_rmse:.4f})")
        final_validated_models[loc_str] = tuned_model
        winner = "Tuned"
        final_rmse = tuned_rmse
    else:
        print(f"Location {loc}: Reverting to Base {SARIMA_ORDER} (RMSE: {base_rmse:.4f} <= Tuned: {tuned_rmse:.4f})")
        final_validated_models[loc_str] = base_model
        winner = "Base"
        final_rmse = base_rmse

    validation_results.append({
        'Location': loc,
        'Base Order': SARIMA_ORDER,
        'Base RMSE': base_rmse,
        'Tuned Order': tuned_order,
        'Tuned RMSE': tuned_rmse,
        'Winner': winner,
        'Final RMSE': final_rmse
    })

print("\nFallback selection complete!")
display(pd.DataFrame(validation_results).round(4))

FALLBACK STRATEGY: Default (1,0,1) vs Tuned Order for Top 20
Location 26125: Keeping Tuned (2, 1, 2) (RMSE: 1319.0243 < Base: 1626.6762)
Location 26163: Keeping Tuned (2, 1, 2) (RMSE: 1077.2712 < Base: 1385.4764)
Location 26099: Keeping Tuned (2, 1, 2) (RMSE: 673.7939 < Base: 786.7982)
Location 26139: Keeping Tuned (2, 1, 2) (RMSE: 711.5255 < Base: 712.5007)
Location 26087: Reverting to Base (1, 0, 1) (RMSE: 606.5252 <= Tuned: 799.9246)
Location 26115: Keeping Tuned (2, 1, 2) (RMSE: 313.0085 < Base: 314.5511)
Location 26021: Keeping Tuned (2, 0, 2) (RMSE: 259.8964 < Base: 260.2300)
Location 26065: Keeping Tuned (2, 0, 2) (RMSE: 222.7545 < Base: 222.7546)
Location 26049: Keeping Tuned (2, 0, 2) (RMSE: 207.6372 < Base: 208.9258)
Location 26093: Keeping Tuned (1, 1, 2) (RMSE: 145.9607 < Base: 146.4675)
Location 26073: Reverting to Base (1, 0, 1) (RMSE: 131.0405 <= Tuned: 142.0400)
Location 26117: Reverting to Base (1, 0, 1) (RMSE: 120.1416 <= Tuned: 120.7226)
Location 26081: Keeping Tuned

,Location,Base Order,Base RMSE,Tuned Order,Tuned RMSE,Winner,Final RMSE
0,26125,"(1, 0, 1)",1626.6762,"(2, 1, 2)",1319.0243,Tuned,1319.0243
1,26163,"(1, 0, 1)",1385.4764,"(2, 1, 2)",1077.2712,Tuned,1077.2712
2,26099,"(1, 0, 1)",786.7982,"(2, 1, 2)",673.7939,Tuned,673.7939
3,26139,"(1, 0, 1)",712.5007,"(2, 1, 2)",711.5255,Tuned,711.5255
4,26087,"(1, 0, 1)",606.5252,"(1, 1, 2)",799.9246,Base,606.5252
5,26115,"(1, 0, 1)",314.5511,"(2, 1, 2)",313.0085,Tuned,313.0085
6,26021,"(1, 0, 1)",260.2300,"(2, 0, 2)",259.8964,Tuned,259.8964
7,26065,"(1, 0, 1)",222.7546,"(2, 0, 2)",222.7545,Tuned,222.7545
8,26049,"(1, 0, 1)",208.9258,"(2, 0, 2)",207.6372,Tuned,207.6372
9,26093,"(1, 0, 1)",146.4675,"(1, 1, 2)",145.9607,Tuned,145.9607


In [38]:
print("="*70)
print("OVERALL VALIDATION EVALUATION - 24H HORIZON (WITH FALLBACK FOR TOP 20)")
print("="*70)

# Generate predictions for all locations using the final validated models
validated_all_preds_24h_df = generate_sarima_predictions(final_validated_models, locations, val_timestamps_24h)

# Evaluate per county
validated_all_rmses = evaluate_per_county(val_truth_24h, validated_all_preds_24h_df, locations)

# Calculate new average RMSE
validated_all_avg = np.nanmean(validated_all_rmses)

print(f"Original Overall Average RMSE (Base models only): {sarima_24h_avg:.4f}")
print(f"New Overall Average RMSE (with Fallback Strategy for Top 20): {validated_all_avg:.4f}")

if validated_all_avg < sarima_24h_avg:
    improvement = (sarima_24h_avg - validated_all_avg) / sarima_24h_avg * 100
    print(f"\n✓ Fallback strategy improved the overall RMSE by {improvement:.2f}%")
elif validated_all_avg > sarima_24h_avg:
    degradation = (validated_all_avg - sarima_24h_avg) / sarima_24h_avg * 100
    print(f"\n✗ Fallback strategy worsened the overall RMSE by {degradation:.2f}%")
else:
    print("\n- No change in overall RMSE.")

# Update sarima_models globally to reflect these fallback choices for downstream tasks
sarima_models = final_validated_models

OVERALL VALIDATION EVALUATION - 24H HORIZON (WITH FALLBACK FOR TOP 20)
Original Overall Average RMSE (Base models only): 89.7450
New Overall Average RMSE (with Fallback Strategy for Top 20): 80.7089

✓ Fallback strategy improved the overall RMSE by 10.07%


### 48H

In [ ]:
import pandas as pd

# Create a DataFrame with locations and their corresponding RMSEs
rmse_df = pd.DataFrame({
    'location': locations,
    'rmse_48h': sarima_48h_rmses
})

# Sort by RMSE descending to find the worst performing counties
top_worst_counties = rmse_df.sort_values(by='rmse_48h', ascending=False).reset_index(drop=True)

print("Top 20 Counties with Highest Validation RMSE (48h Horizon):")
display(top_worst_counties.head(20))

In [ ]:
import itertools
import warnings
warnings.filterwarnings('ignore')

# Extract the top 20 worst locations
top_20_locations = top_worst_counties['location'].head(20).tolist()

print("="*70)
print("Tuning SARIMA (p,d,q) for the Top 20 Worst Locations")
print("="*70)

# Define parameter range for p, d, q
p = q = range(0, 3)
d = [0, 1]
pdq_combinations = list(itertools.product(p, d, q))

best_orders_top20 = {}
sarima_models_48h_top20 = {} # New variable to store tuned models for 48h top 20

for loc in top_20_locations:
    print(f"\nTuning for location {loc}...")
    y_train_loc = ds_train_sub.out.sel(location=loc).values.astype(float).flatten()

    best_aic = float("inf")
    best_order = None
    best_model_res = None

    for order in pdq_combinations:
        try:
            model = SARIMAX(y_train_loc, order=order, enforce_stationarity=False, enforce_invertibility=False)
            res = model.fit(disp=False)
            if res.aic < best_aic:
                best_aic = res.aic
                best_order = order
                best_model_res = res
        except Exception:
            continue

    if best_order is not None:
        print(f"  ✓ Best Order: {best_order} (AIC: {best_aic:.2f})")
        best_orders_top20[loc] = best_order

        # Update the new separate dictionary with the tuned model
        sarima_models_48h_top20[str(loc)] = best_model_res
    else:
        print(f"  ✗ Failed to find a valid order for {loc}")

print("\nTuning complete for top 20 worst locations! Models are stored in `sarima_models_48h_top20`.")

In [ ]:
import pandas as pd

print("Evaluating Tuned vs Original SARIMA for Top 20 Worst Locations (48h Horizon)\n")

# Generate predictions for just the top 20 locations using the updated models
tuned_preds_top20_df = generate_sarima_predictions(
    sarima_models_48h_top20,
    top_20_locations,
    val_timestamps_48h
)

results = []

for loc in top_20_locations:
    # Get original RMSE
    loc_idx = locations.index(loc)
    original_rmse_val = sarima_48h_rmses[loc_idx]

    # Get new predictions
    loc_str = str(loc)
    loc_pred = tuned_preds_top20_df[tuned_preds_top20_df['location'].astype(str) == loc_str]['pred'].values

    # Get truth for this specific location
    truth_loc = val_truth_48h[:, loc_idx]

    # Calculate new RMSE
    if len(loc_pred) == len(truth_loc):
        new_rmse_val = rmse(truth_loc, loc_pred)
    else:
        new_rmse_val = np.nan

    improvement = original_rmse_val - new_rmse_val
    pct_improvement = (improvement / original_rmse_val * 100) if original_rmse_val > 0 else 0

    results.append({
        'Location': loc,
        'Original Order': SARIMA_ORDER,
        'Original RMSE': original_rmse_val,
        'Tuned Order': best_orders_top20.get(loc, "Failed"),
        'Tuned RMSE': new_rmse_val,
        'Improvement (%)': pct_improvement
    })

comparison_df = pd.DataFrame(results)
display(comparison_df.round(4))

In [ ]:
import pandas as pd
import numpy as np

print("="*70)
print("FALLBACK STRATEGY: Default (1,0,1) vs Tuned Order for Top 20 (48H)")
print("="*70)

# We will store the final approved models here
# For 48h, we use a separate dictionary to store models, as per previous decision
final_validated_models_48h = sarima_models.copy() # Start with all default models
validation_results_48h = []

for loc in top_20_locations:
    loc_str = str(loc)
    y_train_loc = ds_train_sub.out.sel(location=loc).values.astype(float).flatten()

    # 1. Base Model (1, 0, 1)
    base_model = safe_fit_sarima(y_train_loc, order=SARIMA_ORDER);

    # 2. Tuned Model (currently stored in sarima_models_48h_top20 for this horizon)
    tuned_model = sarima_models_48h_top20.get(loc_str)
    tuned_order = best_orders_top20.get(loc, "Failed")

    # Extract truth for this location (48h)
    loc_idx = locations.index(loc)
    truth_loc = val_truth_48h[:, loc_idx]

    # Get base prediction & RMSE (48h)
    if base_model is not None:
        base_pred = generate_sarima_predictions({loc_str: base_model}, [loc], val_timestamps_48h)
        base_rmse = rmse(truth_loc, base_pred['pred'].values)
    else:
        base_rmse = float('inf')

    # Get tuned prediction & RMSE (48h)
    if tuned_model is not None:
        tuned_pred = generate_sarima_predictions({loc_str: tuned_model}, [loc], val_timestamps_48h)
        tuned_rmse = rmse(truth_loc, tuned_pred['pred'].values)
    else:
        tuned_rmse = float('inf')

    # Compare & Pick Winner
    if tuned_rmse < base_rmse:
        print(f"Location {loc}: Keeping Tuned {tuned_order} (RMSE: {tuned_rmse:.4f} < Base: {base_rmse:.4f})")
        final_validated_models_48h[loc_str] = tuned_model
        winner = "Tuned"
        final_rmse = tuned_rmse
    else:
        print(f"Location {loc}: Reverting to Base {SARIMA_ORDER} (RMSE: {base_rmse:.4f} <= Tuned: {tuned_rmse:.4f})")
        final_validated_models_48h[loc_str] = base_model
        winner = "Base"
        final_rmse = base_rmse

    validation_results_48h.append({
        'Location': loc,
        'Base Order': SARIMA_ORDER,
        'Base RMSE': base_rmse,
        'Tuned Order': tuned_order,
        'Tuned RMSE': tuned_rmse,
        'Winner': winner,
        'Final RMSE': final_rmse
    })

print("\nFallback selection complete for 48H!")
display(pd.DataFrame(validation_results_48h).round(4))

In [ ]:
print("="*70)
print("OVERALL VALIDATION EVALUATION - 48H HORIZON (WITH FALLBACK FOR TOP 20 COUNTIES)")
print("="*70)

# Generate predictions for all locations using the final validated models for 48H
validated_all_preds_48h_df = generate_sarima_predictions(final_validated_models_48h, locations, val_timestamps_48h)

# Evaluate per county for 48H
validated_all_rmses_48h = evaluate_per_county(val_truth_48h, validated_all_preds_48h_df, locations)

# Calculate new average RMSE for 48H
validated_all_avg_48h = np.nanmean(validated_all_rmses_48h)

print(f"Original Overall Average RMSE (Base models only, 48H): {sarima_48h_avg:.4f}")
print(f"New Overall Average RMSE (with Fallback Strategy for Top 20 Counties, 48H): {validated_all_avg_48h:.4f}")

if validated_all_avg_48h < sarima_48h_avg:
    improvement = (sarima_48h_avg - validated_all_avg_48h) / sarima_48h_avg * 100
    print(f"\n✓ Fallback strategy improved the overall RMSE for 48H by {improvement:.2f}%")
elif validated_all_avg_48h > sarima_48h_avg:
    degradation = (validated_all_avg_48h - sarima_48h_avg) / sarima_48h_avg * 100
    print(f"\n✗ Fallback strategy worsened the overall RMSE for 48H by {degradation:.2f}%")
else:
    print("\n- No change in overall RMSE for 48H.")

# Update sarima_models globally to reflect these fallback choices for 48H for downstream tasks
sarima_models = final_validated_models_48h

## 6. Seasonal orders for top 10 worst counties

### 4.1 24H

In [20]:
import pandas as pd

# Create a DataFrame with locations and their corresponding RMSEs
rmse_df = pd.DataFrame({
    'location': locations,
    'rmse_24h': sarima_24h_rmses
})

# Sort by RMSE descending to find the worst performing counties
top_worst_counties = rmse_df.sort_values(by='rmse_24h', ascending=False).reset_index(drop=True)

print("Top 10 Counties with Highest Validation RMSE (24h Horizon):")
display(top_worst_counties.head(10))

Top 10 Counties with Highest Validation RMSE (24h Horizon):


,location,rmse_24h
0,26125,1626.676181
1,26163,1385.476389
2,26099,786.798246
3,26139,712.500698
4,26087,606.525192
5,26115,314.551063
6,26021,260.230040
7,26065,222.754602
8,26049,208.925794
9,26093,146.467487


In [21]:
import itertools
import warnings
warnings.filterwarnings('ignore')

# Extract the top 10 worst locations
top_10_locations = top_worst_counties['location'].head(10).tolist()

print("="*70)
print("Tuning SARIMA (p,d,q)(P,D,Q,s) for the Top 10 Worst Locations")
print("="*70)

# Define parameter range for non-seasonal p, d, q
p = q = range(0, 3)
d = [0, 1]
pdq_combinations = list(itertools.product(p, d, q))

# Define seasonal orders
# Using s=24 for daily seasonality given hourly data
seasonal_order_list = [
    (1, 0, 0, 24), # User requested
    (0, 1, 1, 24)  # Suggested seasonal order with differencing
]

best_orders_seasonal_top10 = {}
sarima_seasonal_models_top10 = {} # New dictionary to store seasonal models

for loc in top_10_locations:
    print(f"\nTuning for location {loc}...")
    y_train_loc = ds_train_sub.out.sel(location=loc).values.astype(float).flatten()

    best_aic = float("inf")
    best_order = None
    best_seasonal_order = None
    best_model_res = None

    # Iterate through non-seasonal and seasonal combinations
    for order in pdq_combinations:
        for seasonal_order in seasonal_order_list:
            try:
                model = SARIMAX(y_train_loc, order=order, seasonal_order=seasonal_order, enforce_stationarity=False, enforce_invertibility=False)
                res = model.fit(disp=False)
                if res.aic < best_aic:
                    best_aic = res.aic
                    best_order = order
                    best_seasonal_order = seasonal_order
                    best_model_res = res
            except Exception:
                continue

    if best_order is not None:
        print(f"  ✓ Best Order: {best_order}x{best_seasonal_order} (AIC: {best_aic:.2f})")
        best_orders_seasonal_top10[loc] = (best_order, best_seasonal_order)
        sarima_seasonal_models_top10[str(loc)] = best_model_res
    else:
        print(f"  ✗ Failed to find a valid order for {loc}")

print("\nTuning complete for top 10 worst locations with seasonal orders! Models are stored in `sarima_seasonal_models_top10`.")


Tuning SARIMA (p,d,q)(P,D,Q,s) for the Top 10 Worst Locations

Tuning for location 26125...
  ✓ Best Order: (1, 0, 2)x(0, 1, 1, 24) (AIC: 26355.27)

Tuning for location 26163...
  ✓ Best Order: (2, 0, 1)x(0, 1, 1, 24) (AIC: 26116.96)

Tuning for location 26099...
  ✓ Best Order: (1, 1, 2)x(0, 1, 1, 24) (AIC: 25158.39)

Tuning for location 26139...
  ✓ Best Order: (1, 0, 2)x(0, 1, 1, 24) (AIC: 20716.53)

Tuning for location 26087...
  ✓ Best Order: (1, 1, 2)x(0, 1, 1, 24) (AIC: 18452.69)

Tuning for location 26115...
  ✓ Best Order: (2, 0, 2)x(0, 1, 1, 24) (AIC: 20495.41)

Tuning for location 26021...
  ✓ Best Order: (2, 0, 2)x(0, 1, 1, 24) (AIC: 19644.17)

Tuning for location 26065...
  ✓ Best Order: (0, 0, 2)x(0, 1, 1, 24) (AIC: 22402.28)

Tuning for location 26049...
  ✓ Best Order: (2, 0, 2)x(0, 1, 1, 24) (AIC: 21920.78)

Tuning for location 26093...
  ✓ Best Order: (1, 1, 2)x(0, 1, 1, 24) (AIC: 20566.90)

Tuning complete for top 10 worst locations with seasonal orders! Models are s

In [22]:
import pandas as pd

print("Evaluating Tuned Seasonal vs Original SARIMA for Top 10 Worst Locations (24h Horizon)\n")

# Generate predictions for just the top 10 locations using the updated seasonal models
tuned_preds_top10_df = generate_sarima_predictions(
    sarima_seasonal_models_top10,
    top_10_locations,
    val_timestamps_24h
)

results = []

for loc in top_10_locations:
    # Get original RMSE (from non-seasonal SARIMA)
    loc_idx = locations.index(loc)
    original_rmse_val = sarima_24h_rmses[loc_idx]

    # Get new predictions from the tuned seasonal model
    loc_str = str(loc)
    loc_pred = tuned_preds_top10_df[tuned_preds_top10_df['location'].astype(str) == loc_str]['pred'].values

    # Get truth for this specific location
    truth_loc = val_truth_24h[:, loc_idx]

    # Calculate new RMSE
    if len(loc_pred) == len(truth_loc):
        new_rmse_val = rmse(truth_loc, loc_pred)
    else:
        new_rmse_val = np.nan

    improvement = original_rmse_val - new_rmse_val
    pct_improvement = (improvement / original_rmse_val * 100) if original_rmse_val > 0 else 0

    # Get the best seasonal order for display
    tuned_order_seasonal = best_orders_seasonal_top10.get(loc, (("Failed"), ("Failed")))
    formatted_tuned_order = f"{tuned_order_seasonal[0]}x{tuned_order_seasonal[1]}"

    results.append({
        'Location': loc,
        'Original Order': SARIMA_ORDER,
        'Original RMSE': original_rmse_val,
        'Tuned Seasonal Order': formatted_tuned_order,
        'Tuned Seasonal RMSE': new_rmse_val,
        'Improvement (%)': pct_improvement
    })

comparison_df = pd.DataFrame(results)
display(comparison_df.round(4))

Evaluating Tuned Seasonal vs Original SARIMA for Top 10 Worst Locations (24h Horizon)



,Location,Original Order,Original RMSE,Tuned Seasonal Order,Tuned Seasonal RMSE,Improvement (%)
0,26125,"(1, 0, 1)",1626.6762,"(1, 0, 2)x(0, 1, 1, 24)",1318.4116,18.9506
1,26163,"(1, 0, 1)",1385.4764,"(2, 0, 1)x(0, 1, 1, 24)",1039.3377,24.9834
2,26099,"(1, 0, 1)",786.7982,"(1, 1, 2)x(0, 1, 1, 24)",735.3232,6.5423
3,26139,"(1, 0, 1)",712.5007,"(1, 0, 2)x(0, 1, 1, 24)",702.4522,1.4103
4,26087,"(1, 0, 1)",606.5252,"(1, 1, 2)x(0, 1, 1, 24)",895.7700,-47.6888
5,26115,"(1, 0, 1)",314.5511,"(2, 0, 2)x(0, 1, 1, 24)",299.6793,4.7279
6,26021,"(1, 0, 1)",260.2300,"(2, 0, 2)x(0, 1, 1, 24)",257.5568,1.0273
7,26065,"(1, 0, 1)",222.7546,"(0, 0, 2)x(0, 1, 1, 24)",204.6893,8.1099
8,26049,"(1, 0, 1)",208.9258,"(2, 0, 2)x(0, 1, 1, 24)",195.8699,6.2490
9,26093,"(1, 0, 1)",146.4675,"(1, 1, 2)x(0, 1, 1, 24)",144.1485,1.5833


In [23]:
print("="*70)
print("OVERALL VALIDATION EVALUATION - 24H HORIZON (AFTER SEASONAL TUNING TOP 10)")
print("="*70)

# First, calculate the overall RMSE with the currently best non-seasonal models
# (which are in sarima_models after the previous non-seasonal tuning fallback)
non_seasonal_tuned_preds_24h_df = generate_sarima_predictions(sarima_models, locations, val_timestamps_24h)
non_seasonal_tuned_rmses = evaluate_per_county(val_truth_24h, non_seasonal_tuned_preds_24h_df, locations)
non_seasonal_overall_avg = np.nanmean(non_seasonal_tuned_rmses)

print(f"Overall Average RMSE (after non-seasonal tuning fallback): {non_seasonal_overall_avg:.4f}")

# --- Implement a fallback strategy for seasonal models for the top 10 ---
final_seasonal_models = sarima_models.copy() # Start with the current best models

seasonal_fallback_results = []

for loc in top_10_locations:
    loc_str = str(loc)

    # Get the non-seasonal best model from `sarima_models` (after its own fallback)
    non_seasonal_model = sarima_models.get(loc_str)

    # Get the seasonal best model from `sarima_seasonal_models_top10`
    seasonal_model = sarima_seasonal_models_top10.get(loc_str)
    tuned_seasonal_order = best_orders_seasonal_top10.get(loc, (("Failed"), ("Failed")))

    # Calculate RMSE for non-seasonal model
    loc_idx = locations.index(loc)
    truth_loc = val_truth_24h[:, loc_idx]

    if non_seasonal_model is not None:
        non_seasonal_pred_df = generate_sarima_predictions({loc_str: non_seasonal_model}, [loc], val_timestamps_24h)
        non_seasonal_rmse = rmse(truth_loc, non_seasonal_pred_df['pred'].values)
    else:
        non_seasonal_rmse = float('inf')

    # Calculate RMSE for seasonal model
    if seasonal_model is not None:
        seasonal_pred_df = generate_sarima_predictions({loc_str: seasonal_model}, [loc], val_timestamps_24h)
        seasonal_rmse = rmse(truth_loc, seasonal_pred_df['pred'].values)
    else:
        seasonal_rmse = float('inf')

    # Compare and pick the winner for this location
    if seasonal_rmse < non_seasonal_rmse:
        print(f"Location {loc}: Keeping Seasonal {tuned_seasonal_order} (RMSE: {seasonal_rmse:.4f} < Non-Seasonal: {non_seasonal_rmse:.4f})")
        final_seasonal_models[loc_str] = seasonal_model
        winner = "Seasonal"
        final_rmse = seasonal_rmse
    else:
        print(f"Location {loc}: Reverting to Non-Seasonal {SARIMA_ORDER} (RMSE: {non_seasonal_rmse:.4f} <= Seasonal: {seasonal_rmse:.4f})")
        final_seasonal_models[loc_str] = non_seasonal_model
        winner = "Non-Seasonal"
        final_rmse = non_seasonal_rmse

    seasonal_fallback_results.append({
        'Location': loc,
        'Non-Seasonal RMSE': non_seasonal_rmse,
        'Seasonal RMSE': seasonal_rmse,
        'Winner': winner,
        'Final RMSE': final_rmse
    })

print("\nSeasonal Fallback selection complete for Top 10!")
display(pd.DataFrame(seasonal_fallback_results).round(4))

# --- Evaluate overall RMSE with final seasonal models ---
# Generate predictions for all locations using the final validated models
seasonal_validated_all_preds_24h_df = generate_sarima_predictions(final_seasonal_models, locations, val_timestamps_24h)

# Evaluate per county
seasonal_validated_all_rmses = evaluate_per_county(val_truth_24h, seasonal_validated_all_preds_24h_df, locations)

# Calculate new average RMSE
seasonal_validated_all_avg = np.nanmean(seasonal_validated_all_rmses)

print(f"Original Overall Average RMSE (Base models only): {sarima_24h_avg:.4f}")
print(f"Overall Average RMSE (After Non-Seasonal Tuning Fallback): {non_seasonal_overall_avg:.4f}")
print(f"New Overall Average RMSE (After Seasonal Tuning Fallback): {seasonal_validated_all_avg:.4f}")

if seasonal_validated_all_avg < non_seasonal_overall_avg:
    improvement = (non_seasonal_overall_avg - seasonal_validated_all_avg) / non_seasonal_overall_avg * 100
    print(f"\n✓ Seasonal fallback strategy improved the overall RMSE by {improvement:.2f}% (compared to non-seasonal tuning)")
elif seasonal_validated_all_avg > non_seasonal_overall_avg:
    degradation = (seasonal_validated_all_avg - non_seasonal_overall_avg) / non_seasonal_overall_avg * 100
    print(f"\n✗ Seasonal fallback strategy worsened the overall RMSE by {degradation:.2f}% (compared to non-seasonal tuning)")
else:
    print("\n- No change in overall RMSE from seasonal tuning.")

# Update sarima_models globally to reflect these final choices for downstream tasks
sarima_models = final_seasonal_models

OVERALL VALIDATION EVALUATION - 24H HORIZON (AFTER SEASONAL TUNING TOP 10)
Overall Average RMSE (after non-seasonal tuning fallback): 89.7450
Location 26125: Keeping Seasonal ((1, 0, 2), (0, 1, 1, 24)) (RMSE: 1318.4116 < Non-Seasonal: 1626.6762)
Location 26163: Keeping Seasonal ((2, 0, 1), (0, 1, 1, 24)) (RMSE: 1039.3377 < Non-Seasonal: 1385.4764)
Location 26099: Keeping Seasonal ((1, 1, 2), (0, 1, 1, 24)) (RMSE: 735.3232 < Non-Seasonal: 786.7982)
Location 26139: Keeping Seasonal ((1, 0, 2), (0, 1, 1, 24)) (RMSE: 702.4522 < Non-Seasonal: 712.5007)
Location 26087: Reverting to Non-Seasonal (1, 0, 1) (RMSE: 606.5252 <= Seasonal: 895.7700)
Location 26115: Keeping Seasonal ((2, 0, 2), (0, 1, 1, 24)) (RMSE: 299.6793 < Non-Seasonal: 314.5511)
Location 26021: Keeping Seasonal ((2, 0, 2), (0, 1, 1, 24)) (RMSE: 257.5568 < Non-Seasonal: 260.2300)
Location 26065: Keeping Seasonal ((0, 0, 2), (0, 1, 1, 24)) (RMSE: 204.6893 < Non-Seasonal: 222.7546)
Location 26049: Keeping Seasonal ((2, 0, 2), (0, 

,Location,Non-Seasonal RMSE,Seasonal RMSE,Winner,Final RMSE
0,26125,1626.6762,1318.4116,Seasonal,1318.4116
1,26163,1385.4764,1039.3377,Seasonal,1039.3377
2,26099,786.7982,735.3232,Seasonal,735.3232
3,26139,712.5007,702.4522,Seasonal,702.4522
4,26087,606.5252,895.7700,Non-Seasonal,606.5252
5,26115,314.5511,299.6793,Seasonal,299.6793
6,26021,260.2300,257.5568,Seasonal,257.5568
7,26065,222.7546,204.6893,Seasonal,204.6893
8,26049,208.9258,195.8699,Seasonal,195.8699
9,26093,146.4675,144.1485,Seasonal,144.1485


Original Overall Average RMSE (Base models only): 89.7450
Overall Average RMSE (After Non-Seasonal Tuning Fallback): 89.7450
New Overall Average RMSE (After Seasonal Tuning Fallback): 80.5050

✓ Seasonal fallback strategy improved the overall RMSE by 10.30% (compared to non-seasonal tuning)


### 48H

In [24]:
import pandas as pd

# Create a DataFrame with locations and their corresponding RMSEs
rmse_df = pd.DataFrame({
    'location': locations,
    'rmse_48h': sarima_48h_rmses
})

# Sort by RMSE descending to find the worst performing counties
top_worst_counties = rmse_df.sort_values(by='rmse_48h', ascending=False).reset_index(drop=True)

print("Top 10 Counties with Highest Validation RMSE (48h Horizon):")
display(top_worst_counties.head(10))

Top 10 Counties with Highest Validation RMSE (48h Horizon):


,location,rmse_48h
0,26125,1154.226711
1,26163,992.208309
2,26099,569.362720
3,26139,516.697623
4,26087,431.068537
5,26115,222.456527
6,26021,188.639275
7,26065,163.660326
8,26049,162.321998
9,26081,137.590448


In [25]:
import itertools
import warnings
warnings.filterwarnings('ignore')

# Extract the top 10 worst locations
top_10_locations = top_worst_counties['location'].head(10).tolist()

print("="*70)
print("Tuning SARIMA (p,d,q)(P,D,Q,s) for the Top 10 Worst Locations (48h Horizon)")
print("="*70)

# Define parameter range for non-seasonal p, d, q
p = q = range(0, 3)
d = [0, 1]
pdq_combinations = list(itertools.product(p, d, q))

# Define seasonal orders
# Using s=24 for daily seasonality given hourly data
seasonal_order_list = [
    (1, 0, 0, 24), # User requested
    (0, 1, 1, 24)  # Suggested seasonal order with differencing
]

best_orders_seasonal_top10_48h = {} # New variable to store best seasonal orders for 48h
sarima_seasonal_models_48h_top10 = {} # New variable to store tuned seasonal models for 48h top 10

for loc in top_10_locations:
    print(f"\nTuning for location {loc}...")
    y_train_loc = ds_train_sub.out.sel(location=loc).values.astype(float).flatten()

    best_aic = float("inf")
    best_order = None
    best_seasonal_order = None
    best_model_res = None

    for order in pdq_combinations:
        for seasonal_order in seasonal_order_list:
            try:
                model = SARIMAX(y_train_loc, order=order, seasonal_order=seasonal_order, enforce_stationarity=False, enforce_invertibility=False)
                res = model.fit(disp=False)
                if res.aic < best_aic:
                    best_aic = res.aic
                    best_order = order
                    best_seasonal_order = seasonal_order
                    best_model_res = res
            except Exception:
                continue

    if best_order is not None:
        print(f"  ✓ Best Order: {best_order}x{best_seasonal_order} (AIC: {best_aic:.2f})")
        best_orders_seasonal_top10_48h[loc] = (best_order, best_seasonal_order)

        # Update the new separate dictionary with the tuned model
        sarima_seasonal_models_48h_top10[str(loc)] = best_model_res
    else:
        print(f"  ✗ Failed to find a valid order for {loc}")

print("\nTuning complete for top 10 worst locations with seasonal orders! Models are stored in `sarima_seasonal_models_48h_top10`.")

Tuning SARIMA (p,d,q)(P,D,Q,s) for the Top 10 Worst Locations (48h Horizon)

Tuning for location 26125...
  ✓ Best Order: (1, 0, 2)x(0, 1, 1, 24) (AIC: 26355.27)

Tuning for location 26163...
  ✓ Best Order: (2, 0, 1)x(0, 1, 1, 24) (AIC: 26116.96)

Tuning for location 26099...
  ✓ Best Order: (1, 1, 2)x(0, 1, 1, 24) (AIC: 25158.39)

Tuning for location 26139...
  ✓ Best Order: (1, 0, 2)x(0, 1, 1, 24) (AIC: 20716.53)

Tuning for location 26087...
  ✓ Best Order: (1, 1, 2)x(0, 1, 1, 24) (AIC: 18452.69)

Tuning for location 26115...
  ✓ Best Order: (2, 0, 2)x(0, 1, 1, 24) (AIC: 20495.41)

Tuning for location 26021...
  ✓ Best Order: (2, 0, 2)x(0, 1, 1, 24) (AIC: 19644.17)

Tuning for location 26065...
  ✓ Best Order: (0, 0, 2)x(0, 1, 1, 24) (AIC: 22402.28)

Tuning for location 26049...
  ✓ Best Order: (2, 0, 2)x(0, 1, 1, 24) (AIC: 21920.78)

Tuning for location 26081...
  ✓ Best Order: (1, 0, 2)x(0, 1, 1, 24) (AIC: 22979.57)

Tuning complete for top 10 worst locations with seasonal orders

In [26]:
import pandas as pd

print("Evaluating Tuned Seasonal vs Original SARIMA for Top 10 Worst Locations (48h Horizon)\n")

# Generate predictions for just the top 10 locations using the updated seasonal models
tuned_preds_top10_df = generate_sarima_predictions(
    sarima_seasonal_models_48h_top10,
    top_10_locations,
    val_timestamps_48h
)

results = []

for loc in top_10_locations:
    # Get original RMSE
    loc_idx = locations.index(loc)
    original_rmse_val = sarima_48h_rmses[loc_idx]

    # Get new predictions from the tuned seasonal model
    loc_str = str(loc)
    loc_pred = tuned_preds_top10_df[tuned_preds_top10_df['location'].astype(str) == loc_str]['pred'].values

    # Get truth for this specific location
    truth_loc = val_truth_48h[:, loc_idx]

    # Calculate new RMSE
    if len(loc_pred) == len(truth_loc):
        new_rmse_val = rmse(truth_loc, loc_pred)
    else:
        new_rmse_val = np.nan

    improvement = original_rmse_val - new_rmse_val
    pct_improvement = (improvement / original_rmse_val * 100) if original_rmse_val > 0 else 0

    # Get the best seasonal order for display
    tuned_order_seasonal = best_orders_seasonal_top10_48h.get(loc, (("Failed"), ("Failed")))
    formatted_tuned_order = f"{tuned_order_seasonal[0]}x{tuned_order_seasonal[1]}"

    results.append({
        'Location': loc,
        'Original Order': SARIMA_ORDER,
        'Original RMSE': original_rmse_val,
        'Tuned Seasonal Order': formatted_tuned_order,
        'Tuned Seasonal RMSE': new_rmse_val,
        'Improvement (%)': pct_improvement
    })

comparison_df = pd.DataFrame(results)
display(comparison_df.round(4))

Evaluating Tuned Seasonal vs Original SARIMA for Top 10 Worst Locations (48h Horizon)



,Location,Original Order,Original RMSE,Tuned Seasonal Order,Tuned Seasonal RMSE,Improvement (%)
0,26125,"(1, 0, 1)",1154.2267,"(1, 0, 2)x(0, 1, 1, 24)",974.6812,15.5555
1,26163,"(1, 0, 1)",992.2083,"(2, 0, 1)x(0, 1, 1, 24)",775.9690,21.7937
2,26099,"(1, 0, 1)",569.3627,"(1, 1, 2)x(0, 1, 1, 24)",525.2435,7.7489
3,26139,"(1, 0, 1)",516.6976,"(1, 0, 2)x(0, 1, 1, 24)",508.2958,1.6261
4,26087,"(1, 0, 1)",431.0685,"(1, 1, 2)x(0, 1, 1, 24)",1018.7119,-136.3225
5,26115,"(1, 0, 1)",222.4565,"(2, 0, 2)x(0, 1, 1, 24)",212.4635,4.4921
6,26021,"(1, 0, 1)",188.6393,"(2, 0, 2)x(0, 1, 1, 24)",186.2469,1.2683
7,26065,"(1, 0, 1)",163.6603,"(0, 0, 2)x(0, 1, 1, 24)",151.4535,7.4587
8,26049,"(1, 0, 1)",162.3220,"(2, 0, 2)x(0, 1, 1, 24)",155.3844,4.2740
9,26081,"(1, 0, 1)",137.5904,"(1, 0, 2)x(0, 1, 1, 24)",135.4838,1.5311


In [30]:
print("="*70)
print("OVERALL VALIDATION EVALUATION - 48H HORIZON (AFTER SEASONAL TUNING TOP 10)")
print("="*70)

# First, calculate the overall RMSE with the models from the previous non-seasonal fallback stage
# This gives us a baseline to compare against the seasonal tuning results.
non_seasonal_tuned_preds_48h_df = generate_sarima_predictions(sarima_models, locations, val_timestamps_48h)
non_seasonal_tuned_rmses_48h = evaluate_per_county(val_truth_48h, non_seasonal_tuned_preds_48h_df, locations)
non_seasonal_overall_avg_48h = np.nanmean(non_seasonal_tuned_rmses_48h)

print(f"Overall Average RMSE (after non-seasonal tuning fallback, 48H): {non_seasonal_overall_avg_48h:.4f}")

# --- Implement a fallback strategy for seasonal models for the top 10 (48H) ---
# Initialize with the models from the previous non-seasonal fallback stage
final_seasonal_models_48h = sarima_models.copy()

seasonal_fallback_results_48h = []

for loc in top_10_locations:
    loc_str = str(loc)
    y_train_loc = ds_train_sub.out.sel(location=loc).values.astype(float).flatten()

    # 1. Base Model (1,0,1) for this specific location to use as fallback
    base_model_101 = safe_fit_sarima(y_train_loc, order=SARIMA_ORDER)
    base_rmse_101 = float('inf')
    if base_model_101 is not None:
        base_pred_101_df = generate_sarima_predictions({loc_str: base_model_101}, [loc], val_timestamps_48h)
        base_rmse_101 = rmse(val_truth_48h[:, locations.index(loc)], base_pred_101_df['pred'].values)

    # Get the seasonal best model from `sarima_seasonal_models_48h_top10`
    seasonal_model_48h = sarima_seasonal_models_48h_top10.get(loc_str)
    tuned_seasonal_order_48h = best_orders_seasonal_top10_48h.get(loc, (("Failed"), ("Failed")))

    # Calculate RMSE for seasonal model (48H)
    seasonal_rmse_48h = float('inf')
    if seasonal_model_48h is not None:
        seasonal_pred_df_48h = generate_sarima_predictions({loc_str: seasonal_model_48h}, [loc], val_timestamps_48h)
        seasonal_rmse_48h = rmse(val_truth_48h[:, locations.index(loc)], seasonal_pred_df_48h['pred'].values)

    # Compare and pick the winner for this location (48H) using Base(1,0,1) as fallback
    if seasonal_rmse_48h < base_rmse_101:
        print(f"Location {loc}: Keeping Seasonal {tuned_seasonal_order_48h} (RMSE: {seasonal_rmse_48h:.4f} < Base(1,0,1): {base_rmse_101:.4f})")
        final_seasonal_models_48h[loc_str] = seasonal_model_48h
        winner = "Seasonal"
        final_rmse_48h = seasonal_rmse_48h
    else:
        print(f"Location {loc}: Reverting to Base(1,0,1) {SARIMA_ORDER} (RMSE: {base_rmse_101:.4f} <= Seasonal: {seasonal_rmse_48h:.4f})")
        final_seasonal_models_48h[loc_str] = base_model_101 # Store the base (1,0,1) model
        winner = "Base(1,0,1)"
        final_rmse_48h = base_rmse_101

    seasonal_fallback_results_48h.append({
        'Location': loc,
        'Base(1,0,1) RMSE': base_rmse_101, # Changed key to reflect comparison with base
        'Seasonal RMSE': seasonal_rmse_48h,
        'Winner': winner,
        'Final RMSE': final_rmse_48h
    })

print("\nSeasonal Fallback selection complete for Top 10 (48H)!")
display(pd.DataFrame(seasonal_fallback_results_48h).round(4))

# --- Evaluate overall RMSE with final seasonal models (48H) ---
# Generate predictions for all locations using the final validated models for 48H
seasonal_validated_all_preds_48h_df = generate_sarima_predictions(final_seasonal_models_48h, locations, val_timestamps_48h)

# Evaluate per county for 48H
seasonal_validated_all_rmses_48h = evaluate_per_county(val_truth_48h, seasonal_validated_all_preds_48h_df, locations)

# Calculate new average RMSE for 48H
seasonal_validated_all_avg_48h = np.nanmean(seasonal_validated_all_rmses_48h)

# The comparison baseline should now be the overall average RMSE from only base (1,0,1) models.
# sarima_48h_avg already holds the overall average RMSE using only base (1,0,1) models.

print(f"Original Overall Average RMSE (Base models only, 48H): {sarima_48h_avg:.4f}")
print(f"Overall Average RMSE (After Non-Seasonal Tuning Fallback, 48H): {non_seasonal_overall_avg_48h:.4f}") # Keep this for context of previous step
print(f"New Overall Average RMSE (After Seasonal Tuning Fallback): {seasonal_validated_all_avg_48h:.4f}")

if seasonal_validated_all_avg_48h < sarima_48h_avg: # Compare against the true original overall average
    improvement = (sarima_48h_avg - seasonal_validated_all_avg_48h) / sarima_48h_avg * 100
    print(f"\n✓ Seasonal fallback strategy improved the overall RMSE for 48H by {improvement:.2f}% (compared to initial base models)")
elif seasonal_validated_all_avg_48h > sarima_48h_avg:
    degradation = (seasonal_validated_all_avg_48h - sarima_48h_avg) / sarima_48h_avg * 100
    print(f"\n✗ Seasonal fallback strategy worsened the overall RMSE for 48H by {degradation:.2f}% (compared to initial base models)")
else:
    print("\n- No change in overall RMSE from seasonal tuning for 48H (compared to initial base models).")

# Update sarima_models globally to reflect these final choices for downstream tasks
sarima_models = final_seasonal_models_48h

OVERALL VALIDATION EVALUATION - 48H HORIZON (AFTER SEASONAL TUNING TOP 10)
Overall Average RMSE (after non-seasonal tuning fallback, 48H): 62.5465
Location 26125: Keeping Seasonal ((1, 0, 2), (0, 1, 1, 24)) (RMSE: 974.6812 < Base(1,0,1): 1154.2267)
Location 26163: Keeping Seasonal ((2, 0, 1), (0, 1, 1, 24)) (RMSE: 775.9690 < Base(1,0,1): 992.2083)
Location 26099: Keeping Seasonal ((1, 1, 2), (0, 1, 1, 24)) (RMSE: 525.2435 < Base(1,0,1): 569.3627)
Location 26139: Keeping Seasonal ((1, 0, 2), (0, 1, 1, 24)) (RMSE: 508.2958 < Base(1,0,1): 516.6976)
Location 26087: Reverting to Base(1,0,1) (1, 0, 1) (RMSE: 431.0685 <= Seasonal: 1018.7119)
Location 26115: Keeping Seasonal ((2, 0, 2), (0, 1, 1, 24)) (RMSE: 212.4635 < Base(1,0,1): 222.4565)
Location 26021: Keeping Seasonal ((2, 0, 2), (0, 1, 1, 24)) (RMSE: 186.2469 < Base(1,0,1): 188.6393)
Location 26065: Keeping Seasonal ((0, 0, 2), (0, 1, 1, 24)) (RMSE: 151.4535 < Base(1,0,1): 163.6603)
Location 26049: Keeping Seasonal ((2, 0, 2), (0, 1, 1,

,Location,"Base(1,0,1) RMSE",Seasonal RMSE,Winner,Final RMSE
0,26125,1154.2267,974.6812,Seasonal,974.6812
1,26163,992.2083,775.9690,Seasonal,775.9690
2,26099,569.3627,525.2435,Seasonal,525.2435
3,26139,516.6976,508.2958,Seasonal,508.2958
4,26087,431.0685,1018.7119,"Base(1,0,1)",431.0685
5,26115,222.4565,212.4635,Seasonal,212.4635
6,26021,188.6393,186.2469,Seasonal,186.2469
7,26065,163.6603,151.4535,Seasonal,151.4535
8,26049,162.3220,155.3844,Seasonal,155.3844
9,26081,137.5904,135.4838,Seasonal,135.4838


Original Overall Average RMSE (Base models only, 48H): 68.3606
Overall Average RMSE (After Non-Seasonal Tuning Fallback, 48H): 62.5465
New Overall Average RMSE (After Seasonal Tuning Fallback): 62.5465

✓ Seasonal fallback strategy improved the overall RMSE for 48H by 8.51% (compared to initial base models)


# SARIMAX

In [59]:
TRAIN_PATH = os.path.join(DATA_DIR, "train.nc")

In [60]:
# Load datasets
ds_train = xr.open_dataset(TRAIN_PATH)

print(ds_train)

<xarray.Dataset> Size: 159MB
Dimensions:    (location: 83, timestamp: 2161, feature: 109)
Coordinates:
  * location   (location) <U5 2kB '26001' '26003' '26005' ... '26163' '26165'
  * timestamp  (timestamp) datetime64[ns] 17kB 2023-04-01 ... 2023-06-30
  * feature    (feature) object 872B 'SBT113' 'SBT114' 'SBT123' ... 'wz' 'wz_1'
    state      (location) <U2 664B ...
Data variables:
    tracked    (location, timestamp) float64 1MB ...
    out        (location, timestamp) float64 1MB ...
    weather    (location, timestamp, feature) float64 156MB ...
Attributes:
    time_start:  2022-01-01T00:00:00
    time_end:    2022-01-31T23:00:00
    time_now:    2025-07-08T14:59:10


In [61]:
# Create validation split (temporal split)
n_timestamps = len(train_timestamps)
split_idx = int(n_timestamps * (1 - VALIDATION_SPLIT))

# Split the dataset
ds_train_sub = ds_train.isel(timestamp=slice(0, split_idx))
ds_val = ds_train.isel(timestamp=slice(split_idx, None))

train_sub_timestamps = pd.to_datetime(ds_train_sub.timestamp.values)
val_timestamps = pd.to_datetime(ds_val.timestamp.values)

print(f"Training Subset: {len(train_sub_timestamps)} timestamps")
print(f"  Period: {train_sub_timestamps.min()} to {train_sub_timestamps.max()}")
print(f"\nValidation Set: {len(val_timestamps)} timestamps")
print(f"  Period: {val_timestamps.min()} to {val_timestamps.max()}")

# Store validation ground truth for later evaluation
val_truth = ds_val.out.transpose("timestamp", "location").values.astype(float)

Training Subset: 1728 timestamps
  Period: 2023-04-01 00:00:00 to 2023-06-11 23:00:00

Validation Set: 433 timestamps
  Period: 2023-06-12 00:00:00 to 2023-06-30 00:00:00


In [62]:
# Prepare ground truth for last 24 and 48 hours to align with test set
val_truth_24h = ds_val.out.transpose("timestamp", "location").isel(timestamp=slice(0, 24)).values.astype(float)
val_truth_48h = ds_val.out.transpose("timestamp", "location").isel(timestamp=slice(0, 48)).values.astype(float)

print(f"\Validation set shapes:")
print(f"  24h: {val_truth_24h.shape}")
print(f"  48h: {val_truth_48h.shape}")

\Validation set shapes:
  24h: (24, 83)
  48h: (48, 83)


In [63]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.statespace.sarimax import SARIMAX

def build_optionB_by_location(ds_train_sub, ds_val, locations, horizon=24):
    X_train_by_loc = {}
    y_train_by_loc = {}
    X_val_by_loc = {}
    y_val_by_loc = {}

    for loc in locations:
        # train
        weather_train = ds_train_sub['weather'].sel(location=loc).values   # (time, feat)
        out_train = ds_train_sub['out'].sel(location=loc).values.astype(float)

        X_train_rows = []
        y_train_rows = []

        for t in range(len(out_train) - horizon):
            w_t = weather_train[t]
            lag1 = out_train[t]
            lag24 = out_train[t-24] if t >= 24 else 0.0

            features = np.concatenate([w_t, [lag1, lag24]])
            target = out_train[t + horizon]

            X_train_rows.append(features)
            y_train_rows.append(target)

        X_train_by_loc[str(loc)] = np.array(X_train_rows)
        y_train_by_loc[str(loc)] = np.array(y_train_rows)

        # validation
        weather_val = ds_val['weather'].sel(location=loc).values
        out_val = ds_val['out'].sel(location=loc).values.astype(float)

        X_val_rows = []
        y_val_rows = []

        for t in range(len(out_val) - horizon):
            w_t = weather_val[t]
            lag1 = out_val[t]
            lag24 = out_val[t-24] if t >= 24 else 0.0

            features = np.concatenate([w_t, [lag1, lag24]])
            target = out_val[t + horizon]

            X_val_rows.append(features)
            y_val_rows.append(target)

        X_val_by_loc[str(loc)] = np.array(X_val_rows)
        y_val_by_loc[str(loc)] = np.array(y_val_rows)

    return X_train_by_loc, y_train_by_loc, X_val_by_loc, y_val_by_loc

In [52]:
def safe_fit_sarimax_exog(y, X, order=(1, 0, 1)):
    y = np.asarray(y, dtype=float).flatten()
    X = np.asarray(X, dtype=float)

    if len(y) < 8 or np.allclose(y, y[0]):
        return None

    try:
        model = SARIMAX(
            y,
            exog=X,
            order=order,
            enforce_stationarity=False,
            enforce_invertibility=False
        )
        res = model.fit(disp=False)
        return res
    except Exception as e:
        print(f"Warning: SARIMAX fit failed - {str(e)[:80]}")
        return None

In [64]:
H = 24

X_train_by_loc, y_train_by_loc, X_val_by_loc, y_val_by_loc = build_optionB_by_location(
    ds_train_sub, ds_val, locations, horizon=H
)

## Fit scaler per county, train per county, predict per county

In [65]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

def scale_per_location(X_train, X_val, n_components=10):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)

    pca = PCA(n_components=n_components)
    X_train_reduced = pca.fit_transform(X_train_scaled)
    X_val_reduced = pca.transform(X_val_scaled)

    return X_train_reduced, X_val_reduced

In [66]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.decomposition import PCA
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

def fit_predict_optionB_sarimax(
    X_train_by_loc, # Pre-built training features (weather + lagged true outages)
    y_train_by_loc, # Pre-built training targets
    ds_train_sub,   # Original training dataset (filtered if applicable) for initializing lags
    ds_val,         # Original validation dataset (filtered if applicable) for weather features
    locations,
    horizon,        # Prediction horizon (e.g., 24 or 48)
    order=(1, 0, 1),
    n_components=None # Changed default to None
):
    pred_rows = []

    for i, loc in enumerate(locations):
        print(f"[{i+1}/{len(locations)}] Fitting and predicting for location {loc}")
        loc_str = str(loc)

        X_train = X_train_by_loc[loc_str]
        y_train = y_train_by_loc[loc_str]

        if len(X_train) == 0 or np.all(y_train == y_train[0]): # Check for sufficient data and variance
            print(f"Warning: Insufficient or constant training data for {loc_str}. Skipping.")
            for _ in range(horizon):
                 pred_rows.append({"location": loc_str, "pred": 0.0})
            continue

        # --- Scale for training data ---
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)

        # --- Apply PCA conditionally ---
        use_pca = False
        X_train_pca = X_train_scaled # Default to no PCA, if PCA is bypassed

        # Only apply PCA if n_components is an integer, positive, and less than the number of features
        if isinstance(n_components, int) and n_components > 0 and n_components < X_train_scaled.shape[1]:
            use_pca = True
            n_comp = min(n_components, X_train_scaled.shape[1], X_train_scaled.shape[0])
            pca = PCA(n_components=n_comp)
            X_train_pca = pca.fit_transform(X_train_scaled)
        # Otherwise (n_components is None, or not an int, or not positive, or too large), PCA is bypassed.
        # X_train_pca remains X_train_scaled (default).

        # --- Fit SARIMAX model ---
        try:
            model = SARIMAX(
                y_train,
                exog=X_train_pca,
                order=order,
                enforce_stationarity=False,
                enforce_invertibility=False
            )
            res = model.fit(disp=False)
        except Exception as e:
            print(f"Warning: SARIMAX fit failed for {loc_str} - {str(e)[:80]}. Filling with zeros.")
            for _ in range(horizon):
                 pred_rows.append({"location": loc_str, "pred": 0.0})
            continue

        # --- Recursive Prediction ---
        location_preds = []

        # Initialize lagged values from the end of ds_train_sub
        out_data_train_loc = ds_train_sub['out'].sel(location=loc).values
        last_out_train = out_data_train_loc[-1]

        if len(out_data_train_loc) >= 24:
            out_24h_ago_train = out_data_train_loc[-24]
        else:
            out_24h_ago_train = 0.0

        current_lag1 = last_out_train
        current_lag24 = out_24h_ago_train

        for t_val_idx in range(horizon):
            current_weather_features = ds_val['weather'].sel(location=loc).isel(timestamp=t_val_idx).values
            current_features_for_pred = np.concatenate([current_weather_features, [current_lag1, current_lag24]])
            current_exog_reshaped = current_features_for_pred.reshape(1, -1)

            current_exog_scaled = scaler.transform(current_exog_reshaped)

            # Apply PCA transformation if PCA was used during training
            if use_pca:
                current_exog_pca = pca.transform(current_exog_scaled)
            else:
                current_exog_pca = current_exog_scaled

            # Make one-step-ahead prediction
            try:
                pred_step = res.forecast(steps=1, exog=current_exog_pca)[0]
            except Exception as e:
                print(f"Warning: Prediction failed for {loc_str} at step {t_val_idx} - {str(e)[:80]}. Using 0.0.")
                pred_step = 0.0

            pred_step = np.clip(pred_step, 0, None)
            location_preds.append(pred_step)

            current_lag24 = current_lag1
            current_lag1 = pred_step

        for p in location_preds:
            pred_rows.append({
                "location": loc_str,
                "pred": p
            })

    return pd.DataFrame(pred_rows)

## Create predictions

### 24H

In [67]:
pred_df_24 = fit_predict_optionB_sarimax(
    X_train_by_loc,
    y_train_by_loc,
    ds_train_sub,
    ds_val,
    locations,
    horizon=H,
    order=(1, 0, 1),
    n_components=10
)

[1/83] Fitting and predicting for location 26001
[2/83] Fitting and predicting for location 26003
[3/83] Fitting and predicting for location 26005
[4/83] Fitting and predicting for location 26007
[5/83] Fitting and predicting for location 26009
[6/83] Fitting and predicting for location 26011
[7/83] Fitting and predicting for location 26013
[8/83] Fitting and predicting for location 26015
[9/83] Fitting and predicting for location 26017
[10/83] Fitting and predicting for location 26019
[11/83] Fitting and predicting for location 26021
[12/83] Fitting and predicting for location 26023
[13/83] Fitting and predicting for location 26025
[14/83] Fitting and predicting for location 26027
[15/83] Fitting and predicting for location 26029
[16/83] Fitting and predicting for location 26031
[17/83] Fitting and predicting for location 26033
[18/83] Fitting and predicting for location 26035
[19/83] Fitting and predicting for location 26037
[20/83] Fitting and predicting for location 26039
[21/83] F

In [68]:
rmses_24_optionB = evaluate_per_county(val_truth_24h, pred_df_24, locations)
mean_rmse_24_optionB = np.nanmean(rmses_24_optionB)

print("Option B SARIMAX mean RMSE per county:", mean_rmse_24_optionB)

Option B SARIMAX mean RMSE per county: 120.21503556088366


### 48H

In [69]:
pred_df_48 = fit_predict_optionB_sarimax(
    X_train_by_loc,
    y_train_by_loc,
    ds_train_sub,
    ds_val,
    locations,
    horizon=48,
    order=(1, 0, 1),
    n_components=10
)

[1/83] Fitting and predicting for location 26001
[2/83] Fitting and predicting for location 26003
[3/83] Fitting and predicting for location 26005
[4/83] Fitting and predicting for location 26007
[5/83] Fitting and predicting for location 26009
[6/83] Fitting and predicting for location 26011
[7/83] Fitting and predicting for location 26013
[8/83] Fitting and predicting for location 26015
[9/83] Fitting and predicting for location 26017
[10/83] Fitting and predicting for location 26019
[11/83] Fitting and predicting for location 26021
[12/83] Fitting and predicting for location 26023
[13/83] Fitting and predicting for location 26025
[14/83] Fitting and predicting for location 26027
[15/83] Fitting and predicting for location 26029
[16/83] Fitting and predicting for location 26031
[17/83] Fitting and predicting for location 26033
[18/83] Fitting and predicting for location 26035
[19/83] Fitting and predicting for location 26037
[20/83] Fitting and predicting for location 26039
[21/83] F

In [70]:
rmses_48_optionB = evaluate_per_county(val_truth_48h, pred_df_48, locations)
mean_rmse_48_optionB = np.nanmean(rmses_48_optionB)

print("Option B SARIMAX mean RMSE per county (48H):", mean_rmse_48_optionB)

Option B SARIMAX mean RMSE per county (48H): 113.13528739681517


## 10 top features

In [71]:
selected_features = [
    'gh_4',
    'pwat',
    'sh2',
    'cape_1',
    'mslma',
    'sdlwrf',
    'd2m',
    'lcc',
    'tcc',
    'pcdb'
]

ds_train_sub_filtered = ds_train_sub.sel(feature=selected_features)
ds_val_filtered = ds_val.sel(feature=selected_features)

print(f"Original training weather features: {len(ds_train_sub.feature.values)}")
print(f"Filtered training weather features: {len(ds_train_sub_filtered.feature.values)}")

H = 24

X_train_by_loc_filtered, y_train_by_loc_filtered, X_val_by_loc_filtered, y_val_by_loc_filtered = build_optionB_by_location(
    ds_train_sub_filtered, ds_val_filtered, locations, horizon=H
)

Original training weather features: 109
Filtered training weather features: 10


### 24H

In [72]:
pred_df_24_filtered = fit_predict_optionB_sarimax(
    X_train_by_loc_filtered,
    y_train_by_loc_filtered,
    ds_train_sub_filtered,
    ds_val_filtered,
    locations,
    horizon=H,
    order=(1, 0, 1),
    n_components=None # Set to None to bypass PCA for filtered features
)

[1/83] Fitting and predicting for location 26001
[2/83] Fitting and predicting for location 26003
[3/83] Fitting and predicting for location 26005
[4/83] Fitting and predicting for location 26007
[5/83] Fitting and predicting for location 26009
[6/83] Fitting and predicting for location 26011
[7/83] Fitting and predicting for location 26013
[8/83] Fitting and predicting for location 26015
[9/83] Fitting and predicting for location 26017
[10/83] Fitting and predicting for location 26019
[11/83] Fitting and predicting for location 26021
[12/83] Fitting and predicting for location 26023
[13/83] Fitting and predicting for location 26025
[14/83] Fitting and predicting for location 26027
[15/83] Fitting and predicting for location 26029
[16/83] Fitting and predicting for location 26031
[17/83] Fitting and predicting for location 26033
[18/83] Fitting and predicting for location 26035
[19/83] Fitting and predicting for location 26037
[20/83] Fitting and predicting for location 26039
[21/83] F

In [73]:
rmses_24_filtered = evaluate_per_county(val_truth_24h, pred_df_24_filtered, locations)
mean_rmse_24_filtered = np.nanmean(rmses_24_filtered)

print("Option B SARIMAX mean RMSE per county (filtered features):", mean_rmse_24_filtered)

Option B SARIMAX mean RMSE per county (filtered features): 125.90610528836655


### 48H

In [74]:
H = 48

X_train_by_loc_filtered, y_train_by_loc_filtered, X_val_by_loc_filtered, y_val_by_loc_filtered = build_optionB_by_location(
    ds_train_sub_filtered, ds_val_filtered, locations, horizon=H
)

In [75]:
pred_df_48_filtered = fit_predict_optionB_sarimax(
    X_train_by_loc_filtered,
    y_train_by_loc_filtered,
    ds_train_sub_filtered,
    ds_val_filtered,
    locations,
    horizon=H,
    order=(1, 0, 1),
    n_components=None # Set to None to bypass PCA for filtered features
)

[1/83] Fitting and predicting for location 26001
[2/83] Fitting and predicting for location 26003
[3/83] Fitting and predicting for location 26005
[4/83] Fitting and predicting for location 26007
[5/83] Fitting and predicting for location 26009
[6/83] Fitting and predicting for location 26011
[7/83] Fitting and predicting for location 26013
[8/83] Fitting and predicting for location 26015
[9/83] Fitting and predicting for location 26017
[10/83] Fitting and predicting for location 26019
[11/83] Fitting and predicting for location 26021
[12/83] Fitting and predicting for location 26023
[13/83] Fitting and predicting for location 26025
[14/83] Fitting and predicting for location 26027
[15/83] Fitting and predicting for location 26029
[16/83] Fitting and predicting for location 26031
[17/83] Fitting and predicting for location 26033
[18/83] Fitting and predicting for location 26035
[19/83] Fitting and predicting for location 26037
[20/83] Fitting and predicting for location 26039
[21/83] F

In [76]:
rmses_48_filtered = evaluate_per_county(val_truth_48h, pred_df_48_filtered, locations)
mean_rmse_48_filtered = np.nanmean(rmses_48_filtered)

print("Option B SARIMAX mean RMSE per county (filtered features, 48H):", mean_rmse_48_filtered)

Option B SARIMAX mean RMSE per county (filtered features, 48H): 118.48946235824374
